# True / False Experimentation

## Motivation

Anthropic's circuit-tracing work studied how Gemma 2 (2B) answers the open-ended prompt:

> *"Fact: the capital of the state containing Dallas is"*

In that setting the model naturally completes the sentence with a city name, so the token `▁Austin` received ~41 % of the probability mass — a strong, clean signal to trace. Attribution graphs built on the `Austin − Dallas` logit difference were correspondingly rich.

This notebook investigates a **different family of questions**: those that ask the model to judge the truth value of a statement, producing **`True`** or **`False`** as its first token. Empirically, these tokens receive much less probability mass (~5–15 %), and the `True − False` attribution score is correspondingly smaller. Before blaming the model or the methodology, we need to understand **why** — and that is exactly what the three experiments below address.

### Three experiments

| # | Name | What it answers |
|---|------|-----------------|
| 1 | **Baseline softmax** | What is the model's theoretical probability distribution over all tokens for this prompt? |
| 2 | **Empirical sampling** | Does repeated sampling match the softmax prediction? Where do `True` / `False` rank? |
| 3 | **Multi-step rollout** | If the model does not say `True`/`False` at step 1, what does it say — and does a verdict ever appear? |
| 4 | **Attribution graph** | For the `True − False` direction, which internal features drive the model's answer? |

---

**How to change the prompt:** edit the `PROMPT` variable in the **Experiment Configuration** cell below. Every cell downstream uses it automatically.

In [1]:
# One-time dependency bootstrap for a fresh pod/kernel.
# Run this cell once on a new pod, then restart the kernel and run from the top.

# %pip install --no-cache-dir -e /workspace/thesis_circuit_breaker

In [2]:
#@title Runtime + Hugging Face Setup (Local/Runpod) { display-mode: "form" }

import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

print("Configuring local/Runpod environment...")

def _find_repo_root() -> Path | None:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory

    # On Runpod, repo is often under /workspace with different folder names.
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child

    # Optional explicit override for custom locations.
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path

    return None

_root = _find_repo_root()
if _root is not None:
    root_str = str(_root)
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    print(f"Repo root on sys.path: {_root}")

    # Ensure core project dependencies are present in the active kernel env.
    import importlib.util
    import subprocess

    required_modules = [
        "safetensors",
        "transformers",
        "tokenizers",
        "nnsight",
        "transformer_lens",
    ]
    missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
    if missing:
        raise RuntimeError(
            "Missing dependencies detected: "
            f"{missing}. Install once in this kernel env with: `pip install -e {_root}` "
            "then restart kernel and rerun from the top."
        )

    # Pin HF stack to versions compatible with this repo (see pyproject.toml).
    # Newer transformers releases can remove symbols like BertForPreTraining and break lazy imports.
    from importlib.metadata import version as _pkg_version

    def _parse(v: str) -> tuple[int, int, int]:
        core = v.split("+", 1)[0].strip()
        parts = core.split(".")
        nums: list[int] = []
        for p in parts[:3]:
            digits = "".join(ch for ch in p if ch.isdigit())
            nums.append(int(digits) if digits else 0)
        while len(nums) < 3:
            nums.append(0)
        return nums[0], nums[1], nums[2]

    def _lt(a: str, b: str) -> bool:
        return _parse(a) < _parse(b)

    def _gt(a: str, b: str) -> bool:
        return _parse(a) > _parse(b)

    def _gte(v: str, minimum: str) -> bool:
        return not _lt(v, minimum)

    def _pin_hf_stack() -> None:
        try:
            hf_hub_v = _pkg_version("huggingface_hub")
        except Exception:
            hf_hub_v = ""

        try:
            tf_v = _pkg_version("transformers")
        except Exception:
            tf_v = ""

        needs_pin = False
        if hf_hub_v and _gte(hf_hub_v, "1.0.0"):
            needs_pin = True

        if tf_v and (_lt(tf_v, "4.56.0") or _gt(tf_v, "4.57.3")):
            needs_pin = True

        if needs_pin:
            print("Pinning huggingface_hub + transformers to repo-compatible versions...")
            subprocess.check_call(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "--no-cache-dir",
                    "huggingface_hub<1.0.0",
                    "transformers>=4.56.0,<=4.57.3",
                ]
            )
            raise RuntimeError(
                "Pinned huggingface_hub/transformers. Please restart the kernel, then rerun from the top."
            )

    _pin_hf_stack()
else:
    print(
        "Could not locate local circuit_tracer repo. "
        "Clone your repo into /workspace or set CT_REPO_DIR to your repo path."
    )

# Avoid torch import failures after dependency drift in long-lived pod environments.
try:
    from typing_extensions import TypeIs  # type: ignore[attr-defined]
except Exception:
    print("typing_extensions is too old for current torch; upgrading typing_extensions...")
    %pip install -q --no-cache-dir "typing_extensions>=4.12.2"
    raise RuntimeError(
        "Installed typing_extensions>=4.12.2. Please restart the kernel, then rerun from the top."
    )

# Load local .env when available (useful for local runs). On Runpod, env vars are usually set in pod config.
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None and _root is not None:
    _env_file = _root / ".env"
    if _env_file.is_file():
        load_dotenv(_env_file)
        print(f"Loaded {_env_file} (e.g. HF_TOKEN for Hugging Face).")

# Standardize token env names so huggingface_hub can pick it up reliably.
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token and not os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

# Configure persistent cache/output paths when running on Runpod.
if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    cache_dirs = {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
        "HF_DATASETS_CACHE": hf_home / "datasets",
        "TORCH_HOME": persistent_root / "torch",
        "CT_OUTPUT_DIR": persistent_root / "results" / "true_false_exp",
    }
    for key, path in cache_dirs.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)

    print("Runpod detected. Persistent storage configured:")
    print(f"  root={persistent_root}")
    for key in [
        "HF_HOME",
        "HUGGINGFACE_HUB_CACHE",
        "TRANSFORMERS_CACHE",
        "HF_DATASETS_CACHE",
        "TORCH_HOME",
        "CT_OUTPUT_DIR",
    ]:
        print(f"  {key}={os.environ[key]}")

from huggingface_hub import get_token

if get_token() is None:
    print(
        "No Hugging Face token in this environment. google/gemma-2-2b is gated: accept the "
        "license at https://huggingface.co/google/gemma-2-2b then set HF_TOKEN in your env."
    )
    print("If on Runpod: set HF_TOKEN in pod env vars and restart kernel.")
else:
    print("Hugging Face token detected; model/cache should be reusable across runs.")

Configuring local/Runpod environment...
Repo root on sys.path: /workspace/thesis_circuit_breaker
Runpod detected. Persistent storage configured:
  root=/workspace
  HF_HOME=/workspace/hf
  HUGGINGFACE_HUB_CACHE=/workspace/hf/hub
  TRANSFORMERS_CACHE=/workspace/hf/transformers
  HF_DATASETS_CACHE=/workspace/hf/datasets
  TORCH_HOME=/workspace/torch
  CT_OUTPUT_DIR=/workspace/results/true_false_exp
Hugging Face token detected; model/cache should be reusable across runs.


In [3]:
# GPU and environment sanity checks — run at the start of every session.
import os
import torch

print(f"torch version: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Runs will be very slow on CPU.")

for key in [
    "HF_HOME",
    "HUGGINGFACE_HUB_CACHE",
    "TRANSFORMERS_CACHE",
    "HF_DATASETS_CACHE",
    "TORCH_HOME",
    "CT_OUTPUT_DIR",
]:
    print(f"{key}={os.environ.get(key, '<unset>')}")

print("HF token set:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")))

torch version: 2.6.0+cu124
cuda available: True
gpu: NVIDIA RTX A4500
HF_HOME=/workspace/hf
HUGGINGFACE_HUB_CACHE=/workspace/hf/hub
TRANSFORMERS_CACHE=/workspace/hf/transformers
HF_DATASETS_CACHE=/workspace/hf/datasets
TORCH_HOME=/workspace/torch
CT_OUTPUT_DIR=/workspace/results/true_false_exp
HF token set: True


In [4]:
# All imports — collected here so the rest of the notebook stays clean.

import gc
import os
import sys
import json
from pathlib import Path
from collections import Counter

import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.attribution.targets import CustomTarget
from circuit_tracer.utils import create_graph_files
from circuit_tracer.utils.demo_utils import (
    cleanup_cuda,
    display_token_probs,
    display_topk_token_predictions,
    display_top_features_comparison,
    get_top_features,
)

/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Experiment Configuration

**This is the only cell you need to edit** to run the experiments on a different prompt or question.

- Change `PROMPT` to any statement-judgement prompt you want to investigate.
- `TOKEN_TRUE` and `TOKEN_FALSE` are the Gemma tokenizer surface forms for the verdict tokens (the leading `▁` represents a space character — important for Gemma's SentencePiece tokenizer).
- `N_SAMPLES` controls how many times we sample in Experiment 2. 100 is a good starting point; increase to 300+ for tighter empirical estimates.
- `N_STEPS` controls how many tokens the rollout generates in Experiment 3.
- `TRANSCODER_LAZY_ENCODER` — defer loading CLT encoder weights from disk until needed (lowers VRAM spikes during `from_pretrained`).
- `CUDA_MIN_FREE_GIB` — heuristic gate before load (~8 GiB+ often works with `TRANSCODER_LAZY_ENCODER` on a 24 GB GPU with only this notebook). Raise if loads still OOM; set `0.0` to disable the check entirely.

In [ ]:
# ── EXPERIMENT CONFIGURATION ─────────────────────────────────────────────────
# Edit this cell to change the question. Everything downstream uses these variables.


# Verdict tokens (Gemma SentencePiece: leading ▁ = space before the word)
TOKEN_TRUE  = " True"
TOKEN_FALSE = " False"

# Output slug — graph files will be saved under CT_OUTPUT_DIR / OUTPUT_SLUG
OUTPUT_SLUG = "true_false_exp"

# Smaller CUDA footprint while loading transcoder shards (recommended).
TRANSCODER_LAZY_ENCODER = True

# Before loading Gemma + CLT, require roughly this much free VRAM on device 0 (soft guard).
# 8 GiB often suffices with TRANSCODER_LAZY_ENCODER; raise if load still hits CUDA OOM; 0 = skip check.
CUDA_MIN_FREE_GIB = 8.0


## Model Loading

We use `ReplacementModel` from the `circuit_tracer` library. This is a special wrapper around **Gemma 2 (2B)** that substitutes the model's internal MLP layers with a *cross-layer transcoder* (CLT) — a sparse, more interpretable component trained to approximate the same computation.

Why bother? The original MLP neurons are *polysemantic* — each neuron fires for many unrelated concepts at once, making them hard to interpret. CLT features are *sparse* and typically represent a single, human-readable concept (e.g. "triangle geometry", "mathematical inequality"). Attribution graphs computed on this replacement model tell us which such features contributed most to a specific output.

The model weights are downloaded from Hugging Face on first use and cached in `/workspace/hf` on Runpod, so subsequent runs are fast.

In [6]:
# Drop Python references and stale CUDA cache peaks before allocating Gemma + CLT.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    free_gib = free_b / (1024**3)
    total_gib = total_b / (1024**3)
    print(f"CUDA device 0: {free_gib:.2f} GiB free of {total_gib:.2f} GiB total (after gc + empty_cache)")

    if CUDA_MIN_FREE_GIB > 0 and free_gib < CUDA_MIN_FREE_GIB:
        raise RuntimeError(
            f"Not enough free GPU memory to load ReplacementModel reliably "
            f"({free_gib:.2f} GiB free, need roughly >= {CUDA_MIN_FREE_GIB} GiB).\n\n"
            "This usually means another process is using most of the GPU. "
            "Run `nvidia-smi`, stop other Jupyter kernels or Python jobs, restart this kernel, "
            "then rerun from the configuration cell onward. "
            "If you intentionally share GPU, temporarily set CUDA_MIN_FREE_GIB = 0.0 in the experiment config."
        )


CUDA device 0: 19.52 GiB free of 19.70 GiB total (after gc + empty_cache)


In [7]:
model_name       = "google/gemma-2-2b"
transcoder_name  = "gemma"
backend          = "transformerlens"  # change to 'nnsight' for the NNSight backend

from circuit_tracer.utils import get_default_device

device = get_default_device()
print(f"Loading model on {device} (cuda > mps > cpu)")
if device.type == "cpu" and sys.platform == "darwin":
    _mps = getattr(torch.backends, "mps", None)
    _built = _mps.is_built() if _mps is not None else False
    _avail = _mps.is_available() if _mps is not None else False
    print(
        f"macOS: torch.backends.mps.is_built()={_built}, is_available()={_avail}. "
        "If both are False, this PyTorch is likely CPU-only."
    )

model = ReplacementModel.from_pretrained(
    model_name,
    transcoder_name,
    dtype=torch.bfloat16,
    backend=backend,
    device=device,
    lazy_encoder=TRANSCODER_LAZY_ENCODER,
    lazy_decoder=True,
)

tokenizer = model.tokenizer
print("Model loaded.")

Loading model on cuda (cuda > mps > cpu)


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model loaded.


---

## Section 1 — Baseline: What Does the Model Predict?

**Background: softmax and the next-token distribution**

A language model does not produce a single answer — it produces a **probability distribution over the entire vocabulary** (roughly 256,000 tokens for Gemma). This is computed by applying the *softmax* function to a vector of raw scores called *logits*:

$$P(\text{token}_i) = \frac{e^{\text{logit}_i}}{\sum_j e^{\text{logit}_j}}$$

Every token gets a non-zero probability. The model's "answer" is whichever token has the highest probability — but many other tokens also receive significant mass.

**Why True/False are often not at the top**

When asked an open-ended question like *"the capital of Texas is"*, the probability is concentrated on a small set of city names, so `▁Austin` naturally dominates. When asked a judgement question like *"True or False"*, the model must **decide to produce a label** — but:

- `\n` (newline) is often the single most probable token after a question, because many training examples end with a newline.
- The model may have learned to begin an explanation rather than a label (e.g. `▁The`, `▁Yes`).
- Both `▁True` and `▁False` are in the vocabulary, but if instruction-following data is sparse relative to general text, their combined probability may still be small.

This first cell shows the full picture: we run one forward pass, display the top tokens, and highlight where `True` and `False` rank.

In [8]:
prompt1 = "Statement: For any triangle, the sum of any two sides is greater than the third side. Answer:"
prompt2 = "Statement: For any triangle, the sum of any two sides is greater than the third side. Answer with exactly one word: True or False."
prompt3 = "Statement: For any triangle, the sum of any two sides is less than the third side. Answer:"

PROMPT = prompt1

In [9]:
# Resolve vocabulary IDs for the verdict tokens.
idx_true  = tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
idx_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
print(f"{TOKEN_TRUE!r}  → vocab id {idx_true}")
print(f"{TOKEN_FALSE!r} → vocab id {idx_false}")

# Single forward pass to get the logit vector at the last token position.
input_ids = model.ensure_tokenized(PROMPT)
with torch.no_grad():
    baseline_logits, _ = model.get_activations(input_ids)

# Show probabilities for our two verdict tokens.
print("\n--- Verdict token probabilities ---")
display_token_probs(
    baseline_logits,
    token_ids=[idx_true, idx_false],
    labels=[TOKEN_TRUE, TOKEN_FALSE],
    title="Baseline: True vs False",
)

# Show top-15 tokens so we can see where True/False rank among all candidates.
probs_all = torch.softmax(baseline_logits.squeeze(0)[-1].float(), dim=-1)
topk = torch.topk(probs_all, k=15)

print("\n--- Top-15 next tokens ---")
print(f"{'Rank':<6} {'Token':<20} {'Probability':>12} {'Logit':>10}")
print("-" * 52)
for rank, (token_id, prob) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), start=1):
    token_str = tokenizer.decode([token_id])
    logit_val = baseline_logits.squeeze(0)[-1, token_id].item()
    marker = " ◄" if token_id in (idx_true, idx_false) else ""
    print(f"{rank:<6} {repr(token_str):<20} {prob:>11.3%} {logit_val:>10.4f}{marker}")

' True'  → vocab id 5569
' False' → vocab id 7662

--- Verdict token probabilities ---


Token,Probability,Logit
True,14.152%,25.6250
False,3.158%,24.1250



--- Top-15 next tokens ---
Rank   Token                 Probability      Logit
----------------------------------------------------
1      ' True'                  14.152%    25.6250 ◄
2      '\n'                      8.584%    25.1250
3      ' A'                      5.899%    24.7500
4      ' ('                      5.206%    24.6250
5      ' The'                    4.595%    24.5000
6      '\n\n'                    4.595%    24.5000
7      ' If'                     4.595%    24.5000
8      ' '                       3.158%    24.1250
9      ' False'                  3.158%    24.1250 ◄
10     ' Statement'              2.459%    23.8750
11     ' a'                      2.459%    23.8750
12     ' This'                   2.459%    23.8750
13     ' Yes'                    2.459%    23.8750
14     ' Which'                  1.690%    23.5000
15     ' All'                    1.492%    23.3750


---

## Section 2 — Empirical First-Token Distribution

**What are we testing?**

The softmax probabilities above are the model's *theoretical* prediction for the first token. But are they real? One way to verify: **sample the first token repeatedly** and compare the observed frequencies to the predicted probabilities.

If the model assigns `▁True` a 10% probability, we expect it to appear roughly 10 times in 100 independent draws — not exactly 10, because sampling is random, but close.

**How we sample**

We use `torch.multinomial` with **temperature = 1.0**. At this temperature, the sampling distribution is exactly the softmax — no tokens are boosted or suppressed. Each call returns one token drawn proportionally to its probability.

We repeat this `N_SAMPLES` times (default: 100) and count how often each token appeared. This gives us an **empirical distribution** to compare against the theoretical one.

**What to expect**

- For tokens with probability > 10%, the empirical count should be reasonably close to theoretical (within a few percent).
- For tokens with probability < 2%, 100 samples is too few to estimate reliably — we might see 0 counts for tokens that theoretically have non-zero probability.
- If ` True` and ` False` appear rarely in the sample, this is entirely consistent with the small probabilities shown in Section 1 — it is not a bug or a model failure.

In [10]:
# Experiment 2 — empirical sampling
N_SAMPLES   = 1000     # number of independent first-token samples
SAMPLE_TEMP = 1.0     # temperature=1.0 samples exactly from the softmax distribution
SEED        = 42      # for reproducibility


In [11]:
torch.manual_seed(SEED)

# Compute the sampling distribution (temperature scaling, then softmax).
raw_logits_last = baseline_logits.squeeze(0)[-1].float()
scaled_logits   = raw_logits_last / SAMPLE_TEMP  # at temp=1.0 this is a no-op
probs_sample    = torch.softmax(scaled_logits, dim=-1)

# Draw N_SAMPLES tokens from this distribution (with replacement — each draw is independent).
samples = torch.multinomial(probs_sample, num_samples=N_SAMPLES, replacement=True)
counts  = Counter(samples.tolist())

# Convert to a sorted table: (token_str, empirical_count, theoretical_probability)
rows = []
for token_id, count in counts.most_common():
    token_str  = tokenizer.decode([token_id])
    theory_pct = probs_sample[token_id].item() * 100
    empirical_pct = count / N_SAMPLES * 100
    rows.append((token_str, count, empirical_pct, theory_pct))

# Always include the verdict tokens even if they were never sampled.
sampled_ids = set(counts.keys())
for vid, vstr in [(idx_true, TOKEN_TRUE), (idx_false, TOKEN_FALSE)]:
    if vid not in sampled_ids:
        theory_pct = probs_sample[vid].item() * 100
        rows.append((vstr, 0, 0.0, theory_pct))

# Print the first 15 rows only.
print(f"Empirical first-token distribution over {N_SAMPLES} samples (temp={SAMPLE_TEMP}, seed={SEED})")
print("First 15 rows:")
print(f"{'Token':<22} {'Count':>6} {'Empirical %':>12} {'Softmax %':>12} {'Diff':>8}")
print("-" * 64)
for token_str, count, emp_pct, th_pct in sorted(rows, key=lambda r: -r[1])[:20]:
    diff = emp_pct - th_pct
    marker = " ◄" if token_str.strip() in ("True", "False") else ""
    print(f"{repr(token_str):<22} {count:>6} {emp_pct:>11.1f}% {th_pct:>11.3f}% {diff:>+7.1f}%{marker}")

Empirical first-token distribution over 1000 samples (temp=1.0, seed=42)
First 15 rows:
Token                   Count  Empirical %    Softmax %     Diff
----------------------------------------------------------------
' True'                   141        14.1%      14.152%    -0.1% ◄
'\n'                       99         9.9%       8.584%    +1.3%
' ('                       65         6.5%       5.206%    +1.3%
' If'                      53         5.3%       4.595%    +0.7%
' A'                       51         5.1%       5.899%    -0.8%
' '                        41         4.1%       3.158%    +0.9%
' The'                     39         3.9%       4.595%    -0.7%
'\n\n'                     36         3.6%       4.595%    -1.0%
' False'                   28         2.8%       3.158%    -0.4% ◄
' This'                    26         2.6%       2.459%    +0.1%
' a'                       23         2.3%       2.459%    -0.2%
' Statement'               20         2.0%       2.459%    -0.5

---

## Section 3 — Multi-Step Rollout: Where Does the Model "Answer"?

**The question**

Section 2 showed that ` True` and ` False` may not be the most probable *first* token. But maybe the model still gives a correct verdict — just not at position 1. Perhaps it starts with a newline, then `True`. Or perhaps it decides to write a full-sentence explanation first and only labels the answer at the end.

This section investigates by **auto-regressively generating `N_STEPS` tokens** after the prompt, one at a time, and recording the sequence.

Implementation: the code mirrors TransformerLens **`HookedTransformer.generate`** (same **`past_kv_cache`**, greedy argmax each step), but runs a small local loop so we can keep per-step outputs that **`generate`** discards.

**KV caching** recomputes only the *new* token positions each step. A naive loop that reran **`model(full_prefix)`** every time rebuilt logits for **every** position — the unembed produces a **`[batch, seq, vocab]`** tensor each pass and dominated VRAM when headroom was tight.

Some stacks report **mixed BF16/FP32 tensors inside grouped-query attention** when KV caching is enabled; the rollout cell installs a scoped **attention-score matmul in float32** patch so KV rollout stays workable.

Each step prints the **softmax probability of the greedy token** (the probability mass at the argmax). If rollout still hits CUDA **OOM**, free the GPU elsewhere first; lower **`N_STEPS`** if peak memory is insufficient.

**Drivers / containers:** **`nvidia-smi` can report ~full GPU memory yet list no processes.** PyTorch errors may still mention multiple PIDs. Treat “almost no free CUDA memory” as “this session cannot allocate another Gemma-forward peak” until you restart the kernel or pod.

**Greedy decoding**

We use **greedy decoding**: at each step we pick the single most probable next token (argmax of the logits). This is deterministic and easy to trace — the same prompt always produces the same sequence.

Greedy decoding is different from sampling (Section 2): here we always take the *most likely* token, not a randomly drawn one.

**What to look for**

- Does the model produce ` True` or ` False` at all within 15 steps?
- If yes, at what position? And what came before it?
- If no, what kind of continuation does the model generate? (Explanation? Repetition? Formatting?)

The answers inform what attribution target makes sense: if the verdict is at position 3, attributing at position 3 is more meaningful than at position 1.

In [12]:
N_STEPS = 25          # how many tokens to generate after the prompt

In [13]:
print(f"Greedy rollout: generating up to {N_STEPS} new tokens after the prompt.")
print(f"Verdict tokens: {TOKEN_TRUE!r} (id={idx_true}), {TOKEN_FALSE!r} (id={idx_false})")
print("=" * 60)

if torch.cuda.is_available():
    free_gib = torch.cuda.mem_get_info()[0] / (1024**3)
    total_gib = torch.cuda.mem_get_info()[1] / (1024**3)
    print(f"CUDA free before rollout: {free_gib:.2f} / {total_gib:.2f} GiB")
    if free_gib < 3.0:
        print(
            "\n*** WARNING: very little free VRAM. ***\n"
            "PyTorch may still report almost full GPU usage even when `nvidia-smi` hides PIDs "
            "(container/driver views differ). Garbage collection cannot free VRAM owned elsewhere.\n"
            "Restart this kernel alone on the GPU, restart the pod, or set "
            "`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`.\n"
        )

# Same tokenization path as Sections 1–2 (dummy BOS special token handled inside ensure_tokenized).
prompt_batch = model.ensure_tokenized(PROMPT)
if prompt_batch.ndim == 1:
    prompt_batch = prompt_batch.unsqueeze(0)

cleanup_cuda()


def _patch_transformerlens_kv_attn_scores_float32_matmul():
    """Work around mixed BF16/FP32 q vs k inside GQA score matmul when `past_kv_cache` is on.

    Without this, TransformerLens may raise ``expected scalar type Float but found BFloat16``.
    Falling back to ``use_past_kv_cache=False`` replays full context each step and can OOM.
    """
    from transformer_lens.components import abstract_attention as _tl_aa

    cls = _tl_aa.AbstractAttention
    if getattr(cls.calculate_attention_scores, "_ct_kv_bf16_gqa_patch", False):
        return
    _orig = cls.calculate_attention_scores

    def _wrapped(self, q, k):
        return _orig(self, q.float(), k.float()).to(q.dtype)

    _wrapped._ct_kv_bf16_gqa_patch = True  # type: ignore[attr-defined]
    cls.calculate_attention_scores = _wrapped  # type: ignore[method-assign]


_patch_transformerlens_kv_attn_scores_float32_matmul()


def _tl_get_device_for_block_index(index: int, cfg, device=None):
    """TL 3.x removed ``utilities.devices.get_device_for_block_index``; TL 2.x still has it.

    Mirrors TransformerLens semantics for splitting blocks across GPUs (most pods: ``n_devices`` 1).
    """
    import torch

    try:
        from transformer_lens.utilities import devices as _tl_dm

        _fn = getattr(_tl_dm, "get_device_for_block_index", None)
        if _fn is not None:
            return _fn(index, cfg, device)
    except Exception:
        pass

    assert cfg.device is not None
    n_dev = getattr(cfg, "n_devices", 1) or 1
    layers_per_device = cfg.n_layers // n_dev
    if device is None:
        device = cfg.device
    device = torch.device(device)
    if device.type == "cpu":
        return device
    device_index = (device.index or 0) + (
        index // layers_per_device if layers_per_device else 0
    )
    return torch.device(device.type, device_index)


_CT_MIN_HOOKED_KV_CACHE_CLS = None


def _minimal_hooked_kv_cache_class():
    """Vendored KV cache (same behaviour as ``transformer_lens.past_key_value_caching``).

    Some containers ship a broken/incomplete ``transformer_lens`` tree without
    ``past_key_value_caching.py``. This keeps the rollout cell self-contained.
    """
    global _CT_MIN_HOOKED_KV_CACHE_CLS
    if _CT_MIN_HOOKED_KV_CACHE_CLS is not None:
        return _CT_MIN_HOOKED_KV_CACHE_CLS

    from dataclasses import dataclass
    from typing import List

    import torch

    @dataclass
    class _KVEntry:
        past_keys: torch.Tensor
        past_values: torch.Tensor
        frozen: bool = False

        @classmethod
        def init_cache_entry(cls, cfg, device, batch_size: int = 1):
            n_heads = cfg.n_key_value_heads if cfg.n_key_value_heads is not None else cfg.n_heads
            return cls(
                past_keys=torch.empty(
                    (batch_size, 0, n_heads, cfg.d_head), device=device, dtype=cfg.dtype
                ),
                past_values=torch.empty(
                    (batch_size, 0, n_heads, cfg.d_head), device=device, dtype=cfg.dtype
                ),
            )

        def append(self, new_keys, new_values):
            updated_keys = torch.cat([self.past_keys, new_keys], dim=1)
            updated_values = torch.cat([self.past_values, new_values], dim=1)
            if not self.frozen:
                self.past_keys = updated_keys
                self.past_values = updated_values
            return updated_keys, updated_values

    @dataclass
    class _KVCache:
        entries: List[_KVEntry]
        previous_attention_mask: torch.Tensor
        frozen: bool = False

        @classmethod
        def init_cache(cls, cfg, device, batch_size: int = 1):
            return cls(
                entries=[
                    _KVEntry.init_cache_entry(
                        cfg,
                        _tl_get_device_for_block_index(i, cfg, device),
                        batch_size,
                    )
                    for i in range(cfg.n_layers)
                ],
                previous_attention_mask=torch.empty(
                    (batch_size, 0),
                    device=device,
                    dtype=torch.int,
                ),
            )

        def freeze(self):
            self.frozen = True
            for e in self.entries:
                e.frozen = True

        def unfreeze(self):
            self.frozen = False
            for e in self.entries:
                e.frozen = False

        def append_attention_mask(self, attention_mask):
            attention_mask = attention_mask.to(self.previous_attention_mask.device)
            updated_mask = torch.cat([self.previous_attention_mask, attention_mask], dim=-1)
            if not self.frozen:
                self.previous_attention_mask = updated_mask
            return updated_mask

        def __getitem__(self, idx):
            return self.entries[idx]

    _CT_MIN_HOOKED_KV_CACHE_CLS = _KVCache
    return _KVCache


def _tl_hooked_kv_cache_cls():
    """Resolve ``HookedTransformerKeyValueCache`` across TransformerLens installs.

    Prefer the public package export; some environments omit ``past_key_value_caching`` from
    subpackage imports while the file still exists on disk.
    """
    import importlib
    import importlib.util
    import pathlib

    try:
        from transformer_lens import HookedTransformerKeyValueCache as _KV

        return _KV
    except ImportError:
        pass

    try:
        return importlib.import_module(
            "transformer_lens.past_key_value_caching"
        ).HookedTransformerKeyValueCache
    except (ModuleNotFoundError, AttributeError):
        pass

    import transformer_lens as tl

    root = pathlib.Path(tl.__file__).resolve().parent
    path = root / "past_key_value_caching.py"
    if path.is_file():
        spec = importlib.util.spec_from_file_location(
            "transformer_lens.past_key_value_caching",
            path,
        )
        if spec is None or spec.loader is None:
            raise ImportError("Could not load past_key_value_caching spec.") from None
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod.HookedTransformerKeyValueCache

    # Last resort: package on disk omitted past_key_value_caching.py entirely.
    return _minimal_hooked_kv_cache_class()


# ``HookedTransformer.generate`` drops per-step logits. Mirror its KV greedy loop to record
# softmax probability of each greedy token (same forward path, same VRAM profile as generate).


def _greedy_rollout_tokens_and_probs_kv(
    tl_model,
    input_tokens_cpu: torch.Tensor,
    max_new_tokens: int,
) -> tuple[torch.Tensor, list[float]]:
    from transformer_lens import utils as tl_utils

    HookedTransformerKeyValueCache = _tl_hooked_kv_cache_cls()

    prepend_bos = tl_utils.USE_DEFAULT_VALUE
    padding_side = tl_utils.USE_DEFAULT_VALUE

    with tl_utils.LocallyOverridenDefaults(
        tl_model, prepend_bos=prepend_bos, padding_side=padding_side
    ):
        batch_size = input_tokens_cpu.shape[0]
        device = _tl_get_device_for_block_index(0, tl_model.cfg)
        input_on_dev = input_tokens_cpu.to(device)
        past_kv_cache = HookedTransformerKeyValueCache.init_cache(
            tl_model.cfg, tl_model.cfg.device, batch_size
        )
        embeds = tl_model.embed(input_on_dev)
        tl_model.eval()
        sampled_tokens_list: list[torch.Tensor] = []
        step_probs: list[float] = []
        start_at_layer = 0

        for index in range(max_new_tokens):
            pos_offset = tl_model.get_pos_offset(past_kv_cache, batch_size)
            tokens = torch.zeros((embeds.size(0), embeds.size(1)), dtype=torch.int)
            attention_mask = tl_utils.get_attention_mask(
                tl_model.tokenizer,
                tokens,
                False if prepend_bos is None else prepend_bos,
            ).to(device)
            residual, shortformer_pe = tl_model.get_residual(
                embeds,
                pos_offset,
                return_shortformer_pos_embed=True,
                device=device,
                attention_mask=attention_mask,
            )
            if index > 0:
                logits = tl_model.forward(
                    residual[:, -1:],
                    return_type="logits",
                    prepend_bos=prepend_bos,
                    padding_side=padding_side,
                    past_kv_cache=past_kv_cache,
                    start_at_layer=start_at_layer,
                    shortformer_pos_embed=shortformer_pe,
                )
            else:
                logits = tl_model.forward(
                    residual,
                    return_type="logits",
                    prepend_bos=prepend_bos,
                    padding_side=padding_side,
                    past_kv_cache=past_kv_cache,
                    start_at_layer=start_at_layer,
                    shortformer_pos_embed=shortformer_pe,
                )
            final_logits = logits[:, -1, :].float()
            probs = torch.softmax(final_logits, dim=-1)
            sampled = final_logits.argmax(-1).to(
                _tl_get_device_for_block_index(0, tl_model.cfg)
            )
            step_probs.append(probs[0, sampled[0]].item())
            sampled_tokens_list.append(sampled.unsqueeze(1))
            embeds = torch.hstack(
                [embeds, tl_model.embed(sampled.unsqueeze(-1))]
            )

        new_tok = torch.cat(sampled_tokens_list, dim=1)
        full = torch.cat((input_on_dev, new_tok), dim=1)
        return full, step_probs


with torch.inference_mode():
    full_ids, greedy_step_probs = _greedy_rollout_tokens_and_probs_kv(
        model, prompt_batch, N_STEPS
    )

cleanup_cuda()

prefix_len = prompt_batch.shape[1]
new_ids_t = full_ids[0, prefix_len:]
generated: list[tuple[int, int, str, float]] = []
for step, (next_id, prob) in enumerate(
    zip(new_ids_t.tolist(), greedy_step_probs, strict=True),
    start=1,
):
    next_str = tokenizer.decode([next_id])
    generated.append((step, next_id, next_str, prob))

# Display the rollout table.
print(f"{'Step':<6} {'Token':<22} {'Prob':>8}  Note")
print("-" * 55)
for step, token_id, token_str, prob in generated:
    note = ""
    if token_id == idx_true:
        note = "  ◄◄ VERDICT: True"
    elif token_id == idx_false:
        note = "  ◄◄ VERDICT: False"
    prob_cell = f"{prob:>7.3%}"
    print(f"{step:<6} {repr(token_str):<22} {prob_cell}{note}")

# Reconstruct and print the full continuation.
continuation = "".join(t for _, _, t, _ in generated)
print("\n--- Full continuation ---")
print(repr(continuation))

# Summary: did a verdict token appear?
verdict_steps = [
    (step, token_str)
    for step, token_id, token_str, _ in generated
    if token_id in (idx_true, idx_false)
]
print("\n--- Verdict summary ---")
if verdict_steps:
    for step, token_str in verdict_steps:
        print(f"Verdict token {token_str!r} appeared at step {step}.")
else:
    print(f"No verdict token appeared within {N_STEPS} steps.")

Greedy rollout: generating up to 25 new tokens after the prompt.
Verdict tokens: ' True' (id=5569), ' False' (id=7662)
CUDA free before rollout: 10.09 / 19.70 GiB


/tmp/ipykernel_269/2526134376.py:223: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import utils as tl_utils


Step   Token                      Prob  Note
-------------------------------------------------------
1      ' True'                13.941%  ◄◄ VERDICT: True
2      ' or'                  26.148%
3      ' False'               83.654%  ◄◄ VERDICT: False
4      '?'                    45.609%
5      '\n\n'                 50.528%
6      'Step'                 31.369%
7      ' '                    99.976%
8      '1'                    99.990%
9      '\n'                   99.009%
10     '1'                    99.976%
11     ' of'                  99.998%
12     ' '                    99.996%
13     '2'                    57.794%
14     '\n\n'                 95.243%
15     'The'                  21.878%
16     ' statement'           47.095%
17     ' is'                  85.435%
18     ' true'                29.673%
19     '.'                    60.258%
20     '\n\n'                 53.583%
21     'Let'                   9.060%
22     ' $'                   34.201%
23     'a'                

For this prompt, the most likely first token was already "False." Always choosing the most likely token continued the sentence in a somewhat meaningful way, but since the verdict appeared in the first token position, we didn't learn much.

This contrasts with the prompt-2 version, where recursively choosing the most likely token resulted in a less sensible output. Based on that case, I'd drop this approach.

---

## Section 4 — Attribution Graph Analysis

**What is an attribution graph?**

An attribution graph is a directed graph that describes, for a specific prompt and a specific output direction, **which internal features of the model were responsible for producing that output** — and how they influenced each other.

- **Nodes** represent: active CLT features (interpretable sparse components), input token embeddings, reconstruction error terms, and output logits.
- **Edges** represent direct linear effects: how strongly does feature A influence feature B, or feature A influence the final logit?

The graph is built by running the *local replacement model* on the prompt, freezing attention patterns and normalisation denominators, and then computing partial derivatives (attributions) of each output logit with respect to each upstream node.

**Why the True − False contrast?**

We attribute toward the **difference direction** `logit(True) − logit(False)`. This is more informative than attributing to either token alone:

- Features that promote `True` *and* `False` equally will have near-zero attribution to this direction — they are not driving the verdict either way.
- Only features that genuinely tip the balance toward one verdict over the other will appear with high attribution.

In practice, because both `True` and `False` receive small probability mass, the absolute attribution scores will be small compared to the capital-city case. This is expected — a small signal does not mean there is no circuit, it means the circuit's effect is diluted across a broad softmax distribution.

**Interpreting the results**

After running attribution, we:
1. Save the graph files to disk so they can be opened in the `circuit-tracer` local server or on Neuronpedia.
2. Display the top-10 features ranked by total multi-hop influence on the `True − False` direction, with clickable Neuronpedia links.

In [14]:
# Experiment 4 — attribution
ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=True,
)

In [15]:
# Run attribution toward True and False simultaneously.
# Passing both tokens as string targets lets the library compute the contrast direction internally.
print(f"Running attribution for: {TOKEN_TRUE!r} and {TOKEN_FALSE!r}")
print(f"Prompt:", PROMPT)
print()

graph_tf = attribute(
    prompt=PROMPT,
    model=model,
    attribution_targets=[TOKEN_TRUE, TOKEN_FALSE],
    **ATTR_KWARGS,
)

n_features = graph_tf.active_features.shape[0]
n_targets  = len(graph_tf.logit_targets)
print(f"\nAttribution complete: {n_targets} targets, {n_features} active features.")

# Show the logit probabilities assigned to each target.
print("\nTarget probabilities used for attribution weighting:")
for i, (target, prob) in enumerate(zip(graph_tf.logit_targets, graph_tf.logit_probabilities.tolist())):
    print(f"  [{i}] {target.token_str!r:30s}  prob = {prob:.4f}")

Running attribution for: ' True' and ' False'
Prompt: Statement: For any triangle, the sum of any two sides is greater than the third side. Answer:



Phase 0: Precomputing activations and vectors
Precomputation completed in 4.81s
Found 23758 active features
Phase 1: Running forward pass
Forward pass completed in 0.46s
Phase 2: Building input vectors
Using 2 specified logit targets with cumulative probability 0.1709
Will include 8192 of 23758 feature nodes
Input vectors built in 5.37s
Phase 3: Computing logit attributions
Logit attributions completed in 0.33s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:09<00:00, 837.85it/s]
Feature attributions completed in 9.78s
Attribution completed in 24.03s



Attribution complete: 2 targets, 23758 active features.

Target probabilities used for attribution weighting:
  [0] ' True'                         prob = 0.1396
  [1] ' False'                        prob = 0.0311


In [16]:
# Save graph files so they can be visualised in the local circuit-tracer server.
#
# After this cell runs, start the server from a terminal:
#   circuit-tracer server --graph_file_dir <output_dir>
# then open the printed URL in your browser.

output_dir = Path(os.environ.get("CT_OUTPUT_DIR", "results")) / OUTPUT_SLUG
output_dir.mkdir(parents=True, exist_ok=True)

create_graph_files(
    graph_or_path=graph_tf,
    slug=OUTPUT_SLUG,
    output_path=str(output_dir),
    node_threshold=0.8,
    edge_threshold=0.98,
)

print(f"Graph files saved to: {output_dir}")
print(f"To visualise, run:")
print(f"  circuit-tracer server --graph_file_dir {output_dir}")

Graph files saved to: /workspace/results/true_false_exp/true_false_exp
To visualise, run:
  circuit-tracer server --graph_file_dir /workspace/results/true_false_exp/true_false_exp


In [17]:
# Display the top-10 most influential features for the True/False attribution.
#
# Each feature is identified by (layer, token_position, feature_index).
# The Neuronpedia links open an interactive dashboard showing which texts activated that feature,
# helping you interpret what concept the feature represents.

features_tf, scores_tf = get_top_features(graph_tf, n=10)

print(f"Top-10 features by multi-hop influence on {TOKEN_TRUE!r} / {TOKEN_FALSE!r}:")
print(f"{'#':<4} {'(layer, pos, feat_idx)':<30} {'Score':>10}  Neuronpedia")
print("-" * 80)
np_model = "gemma-2-2b"
np_set   = "gemmascope-transcoder-16k"
for i, ((layer, pos, feat_idx), score) in enumerate(zip(features_tf, scores_tf), start=1):
    np_url = f"https://www.neuronpedia.org/{np_model}/{layer}-{np_set}/{feat_idx}"
    print(f"{i:<4} ({layer:>2}, {pos:>3}, {feat_idx:>6})          {score:>10.5f}  {np_url}")

# Also render the styled HTML table (clickable links).
display_top_features_comparison(
    feature_sets={f"{TOKEN_TRUE} / {TOKEN_FALSE}": features_tf},
    scores_sets={f"{TOKEN_TRUE} / {TOKEN_FALSE}": scores_tf},
    neuronpedia_model=np_model,
    neuronpedia_set=np_set,
)

cleanup_cuda()

Top-10 features by multi-hop influence on ' True' / ' False':
#    (layer, pos, feat_idx)              Score  Neuronpedia
--------------------------------------------------------------------------------
1    (16,  21,  14143)             0.00280  https://www.neuronpedia.org/gemma-2-2b/16-gemmascope-transcoder-16k/14143
2    (20,  21,  12997)             0.00261  https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-transcoder-16k/12997
3    (24,  21,   4640)             0.00253  https://www.neuronpedia.org/gemma-2-2b/24-gemmascope-transcoder-16k/4640
4    (18,  21,  15770)             0.00221  https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-transcoder-16k/15770
5    (17,  21,   6448)             0.00161  https://www.neuronpedia.org/gemma-2-2b/17-gemmascope-transcoder-16k/6448
6    ( 0,   2,  16200)             0.00160  https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/16200
7    (18,  21,   2940)             0.00155  https://www.neuronpedia.org/gemma-2-2b/18-gemmas

#,Node,Score
1,"(16, 21, 14143)",0.0028
2,"(20, 21, 12997)",0.0026
3,"(24, 21, 4640)",0.0025
4,"(18, 21, 15770)",0.0022
5,"(17, 21, 6448)",0.0016
6,"(0, 2, 16200)",0.0016
7,"(18, 21, 2940)",0.0015
8,"(19, 21, 6538)",0.0015
9,"(21, 21, 7787)",0.0012
10,"(17, 21, 15506)",0.0012


In [18]:
# Tokenized PROMPT — maps attribution table `pos` to tokenizer pieces.
# Uses the same path as Sections 1–3 (`ensure_tokenized`), matching graph positions.

_prompt_row = model.ensure_tokenized(PROMPT)
_prompt_ids = _prompt_row.squeeze(0) if _prompt_row.ndim == 2 else _prompt_row

print("Tokenized PROMPT — pos → token_id → decoded")
print(repr(PROMPT[:120]) + ("..." if len(PROMPT) > 120 else ""))
print("=" * 64)
print(f"{'pos':>4}  {'token_id':>10}   decoded")
print("-" * 64)
for pos, tid in enumerate(_prompt_ids.tolist()):
    print(f"{pos:>4}  {tid:>10}   {repr(tokenizer.decode([tid]))}")
print("=" * 64)
print(f"Total tokens: {len(_prompt_ids)}  (prompt positions run 0 .. {len(_prompt_ids) - 1}).")

Tokenized PROMPT — pos → token_id → decoded
'Statement: For any triangle, the sum of any two sides is greater than the third side. Answer:'
 pos    token_id   decoded
----------------------------------------------------------------
   0           2   '<bos>'
   1       12195   'Statement'
   2      235292   ':'
   3        1699   ' For'
   4        1089   ' any'
   5       23022   ' triangle'
   6      235269   ','
   7         573   ' the'
   8        2707   ' sum'
   9         576   ' of'
  10        1089   ' any'
  11        1378   ' two'
  12       10012   ' sides'
  13         603   ' is'
  14        6561   ' greater'
  15        1178   ' than'
  16         573   ' the'
  17        4906   ' third'
  18        2857   ' side'
  19      235265   '.'
  20       10358   ' Answer'
  21      235292   ':'
Total tokens: 22  (prompt positions run 0 .. 21).


### Reading the attribution graph with Neuronpedia and token positions**

The **attribution graph** you saved is a compact picture of *which learned features* in the model’s residual stream tend to push the next-token distribution toward “True” versus “False” for this prompt. Nodes are typically **(layer, position, feature index)** triples; directed edges summarize how influence flows between them under the chosen attribution method. The graph is *not* a causal proof of a single mechanism—it is a **hypothesis map**: it tells you where to look when you open the same features in a dedicated viewer or in [Neuronpedia](https://www.neuronpedia.org/).

**Neuronpedia** catalogs **sparse interpretability dictionaries** from several training setups—not only **sparse autoencoders (SAEs)**, which typically **reconstruct** the same activation they encode, but also **sparse transcoders** (e.g. Gemma Scope’s `gemmascope-transcoder-16k`), which approximate **MLP input → MLP output** via a sparse latent rather than vanilla autoencoding. **The links in this notebook use those transcoder dictionaries.** Regardless of variant, each feature id may have a short **label** and **maximum-activation examples**—roughly, “this direction fires on text that looks like …”. That is a **semantic prior**, not a guarantee on every prompt.

The integer **`pos`** in the attribution table is **not** a character offset in the string; it is the **index of the token slot** in the prompt sequence, aligned with how the tokenizer split the text. Run the preceding cell’s printout alongside the feature table: **`pos`** → **`decoded`** lets you anchor each row to **which piece of surface text** the feature attends to or writes through at that depth (early tokens like “Statement … :”, the geometric claim, “Answer … :”, or the **`True` / `False`** region where those tokens appear). Patterns often cluster on **delimiter tokens** (`:`, commas, final `.`), on **True/False tails**, or on an **`Answer:`** slot — wherever task format overlaps with verdict semantics.

**Attribution scores are on an arbitrary relative scale** and shrink or grow with how much probability mass sits on `True` / `False` at the attribution position and how sharp the **`True − False`** logit contrast is. A **weak** verdict (tiny target probs, flat contrast) often yields scores around **10⁻⁴**; if the model **confidently** leans toward one label, the same pipeline can produce **10⁻³**-scale influences without any bug. The graph and Neuronpedia answer: *where* those influences concentrate along the prompt, and *what kind of feature* they are—then you intervene or compare prompts to stress-test that story.

### Interpretation for *this* prompt and run (`prompt1`)

The active configuration is **`PROMPT = prompt1`** — geometry text ending with **`Answer:`** only (no explicit “True or False” tail). In the saved run, **` True`** had about **14%** and **` False`** about **3%** of next-token mass at that position — a **much stronger tilt toward True** than the long instruction-following variant. Matching that, multi-hop scores sit around **\(1.2\times10^{-3}\)–\(2.8\times10^{-3}\)** (~one order of magnitude larger than the flattened distribution on the constrained prompt).

| # | Score | `(layer, pos, feat)` | Neuronpedia |
|---:|---:|---|---|
| 1 | 0.00280 | (16, 21, 14143) | [14143](https://www.neuronpedia.org/gemma-2-2b/16-gemmascope-transcoder-16k/14143) |
| 2 | 0.00261 | (20, 21, 12997) | [12997](https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-transcoder-16k/12997) |
| 3 | 0.00253 | (24, 21, 4640) | [4640](https://www.neuronpedia.org/gemma-2-2b/24-gemmascope-transcoder-16k/4640) |
| 4 | 0.00221 | (18, 21, 15770) | [15770](https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-transcoder-16k/15770) |
| 5 | 0.00161 | (17, 21, 6448) | [6448](https://www.neuronpedia.org/gemma-2-2b/17-gemmascope-transcoder-16k/6448) |
| 6 | 0.00160 | (0, 2, 16200) | [16200](https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/16200) |
| 7 | 0.00155 | (18, 21, 2940) | [2940](https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-transcoder-16k/2940) |
| 8 | 0.00150 | (19, 21, 6538) | [6538](https://www.neuronpedia.org/gemma-2-2b/19-gemmascope-transcoder-16k/6538) |
| 9 | 0.00122 | (21, 21, 7787) | [7787](https://www.neuronpedia.org/gemma-2-2b/21-gemmascope-transcoder-16k/7787) |
| 10 | 0.00122 | (17, 21, 15506) | [15506](https://www.neuronpedia.org/gemma-2-2b/17-gemmascope-transcoder-16k/15506) |


#### Where in the sentence do influences sit?

Cross-reference the token printout immediately above: **most top rows use `pos = 21`**, i.e. the **final `:`** after `Answer`. Only **rank 6** is “early”: **`pos = 2`**, the **first `:`** after `Statement`. So this contrast is overwhelmingly attributed to features acting through the **response slot**, not uniformly across `triangle … third side.` A triangle-specific representation *could* still live lower in the graph or below the pruning threshold—but **nothing in positions 5–18** (including `triangle`, `sides`, `greater`) appears among these ten.

#### Neuronpedia glosses vs. triangles / inequalities

[14143](https://www.neuronpedia.org/gemma-2-2b/16-gemmascope-transcoder-16k/14143), [15770](https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-transcoder-16k/15770), [4640](https://www.neuronpedia.org/gemma-2-2b/24-gemmascope-transcoder-16k/4640), [6448](https://www.neuronpedia.org/gemma-2-2b/17-gemmascope-transcoder-16k/6448) read as **truth / validation / factual-assertion** language in Neuronpedia’s auto labels — consistent with **` True`** dominating the softmax, not specifically with Euclidean geometry vocabulary.

[12997](https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-transcoder-16k/12997), [16200](https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/16200), [15506](https://www.neuronpedia.org/gemma-2-2b/17-gemmascope-transcoder-16k/15506) tilt toward **code / syntax / booleans / conditionals**. That mirrors many math-word problems appearing as **pseudo-code or exam templates** (“Statement: … Answer:”). It does **not** license a literal claim that Gemma parsed a Euclidean diagram; rather, **format + boolean semantics** dominate the dashboards for these ids.

[6538](https://www.neuronpedia.org/gemma-2-2b/19-gemmascope-transcoder-16k/6538): **question–answer** framing (“yes”). [2940](https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-transcoder-16k/2940): clauses about **whether something holds vs. explicit false statements**.

**Closest to substantive math wording:** [7787](https://www.neuronpedia.org/gemma-2-2b/21-gemmascope-transcoder-16k/7787) is labeled with **equations / logical deductions** (among other motifs). That is the most plausible **tangential bridge** toward content like inequalities — still **orthogonal to “triangle geometry” proper** in the Neuronpedia blurbs; inspect its **maximum-activating examples** yourself to judge whether elementary geometry triggers show up beyond abstract “math-ish” prose.

### Takeaway

Higher scores here mainly reflect **a clearer verdict direction** (`True ≫ False`). Mechanistically, **open the clickable links above** — the dashboards show firing examples rather than summaries alone. Expect the story to emphasize **truth-style and Q/A-format cues at `Answer:`**; **do not** expect the top‑10 slice to cleanly spell “triangle inequality” unless you expand the subgraph or widen `n`/`max_feature_nodes`. Rerun attribution after changing **`PROMPT`** and revise this prose to match.


---
## Section 5 — Graph diagnostics and interventions

The ranked feature table is a starting point. This section adds six concrete steps that can
turn the attribution graph into *evidence* rather than *correlation*:

1. **Heavy pruning** — saves a much more compact graph JSON that you can comfortably inspect
   in the local `circuit-tracer` server or on Neuronpedia without being overwhelmed by thousands
   of low-weight nodes.
2. **How many nodes survive?** — a quick diagnostic that shows how the node count shrinks as
   you raise the threshold, helping you pick a useful view.
3. **Contrastive graph** — run attribution again on an *altered* prompt (same instruction,
   different geometry claim that is *false*) and compare the top features. Features that persist
   across both prompts signal *verdict format* cues; features that shift reveal *content-sensitive*
   circuit paths.
4. **Feature zero-ablation** — silence the highest-scoring feature and measure the change in
   `logit(True) − logit(False)`. A real mechanistic feature should cause the gap to shrink.
5. **§5e — Intersection sign-flips** — merge **all** selected `(layer, pos, feat)` across true/false
   claims; keep opposite-signed rows in `pos∈[5,18]`; rank by **`|Δ|`** with Neuronpedia links.
6. **§5f — Geometry hooks** — resolve §**5a** gallery to `(layer, pos, feat)` (`pos∈[5,18]`), both graphs, ranked by **`|Δ|`**.


### 5a — Heavily pruned graph (node_threshold=0.6, edge_threshold=0.99)

The default `create_graph_files` call uses `node_threshold=0.8`; raising it to **0.6** keeps
only nodes in the top-60 % of cumulative influence — typically a handful of dozen nodes rather
than hundreds — giving a much cleaner picture in the interactive viewer.

> **To view:** after running the cell below, start the local server on RunPod:
> ```
> circuit-tracer start-server --graph_file_dir <output_dir> --port 8041
> ```
> then open the forwarded port in your browser.


In [19]:
from circuit_tracer.utils.create_graph_files import create_graph_files
import os

# Compact slug so the pruned version doesn't overwrite the original
pruned_slug = OUTPUT_SLUG + "_pruned60"

create_graph_files(
    graph_or_path=graph_tf,
    slug=pruned_slug,
    output_path=str(output_dir),
    node_threshold=0.6,   # keep top-60 % of cumulative influence
    edge_threshold=0.99,  # keep top-1 % of edges by influence
)

print(f"Heavily pruned graph saved to: {output_dir}")
print(f"Slug: {pruned_slug}")
print(f"JSON size: {os.path.getsize(os.path.join(str(output_dir), pruned_slug + '.json')) / 1024:.1f} KB")


Heavily pruned graph saved to: /workspace/results/true_false_exp/true_false_exp
Slug: true_false_exp_pruned60
JSON size: 9131.3 KB


#### Interactive viewer spot-check — when geometry shows up in the subgraph

**Why pruning exists.** Attribution produces a dense web: many transcoder features can activate on one prompt, and tracing every edge is beyond what a plot (or human attention span) tolerates.
**Pruning** keeps the picture readable by trimming weak nodes and edges. The trade-off is blunt: tightening removes clutter but can hide directions that matter only mildly for the plotted contrast yet still carry interpretable semantics—often **near the keyword you care about**.

**Two knobs with similar names.** The word “pruning” appears in **two setups** in this workflow; confusing them explains half the ambiguity when reading notebooks side by side with screenshots.

| Where | What it does |
|---|---|
| **This notebook (`create_graph_files` in Section 5a)** | Sets `node_threshold` (plus `edge_threshold`) when writing a **JSON subgraph** for the server—for example **`0.6`** keeps heavier cumulative-influence trimming so the exported file stays compact. |
| **The browser UI** | Usually exposes its own **pruning/influence slider** over whatever graph JSON you loaded. That slider controls **visibility on screen**. **Looser** settings bring back borderline influencers so you can **hunt visually** below the cutoff of a strict export. |

**Slider numbers versus notebook numbers.** Two different controls can both be called “pruning,” but **they measure things differently.**

- What you dial in Python (`node_threshold` when exporting JSON) is **one machinery**.
- What the browser shows (**e.g.** `0.80` on a slider) is **another**. You should **not** read `0.8` there as “the same as” `node_threshold = 0.8`.

For reproducibility it is safer to describe **what you saw** (“we eased the slider until triangle-related nodes stayed visible”), not **numeric equality** between UI and notebook.

**What the screenshots illustrate.** Relaxing pruning in the viewer is exploratory: you widen the funnel so faint paths draw on screen.

- **Where we hovered:** screenshots were taken with the slider **around ~0.80**; neighbouring settings **~0.76–0.80** often showed the **same qualitative story** (exact cutoffs depend on build and saved graph).
- **With strict pruning**, the subgraph often emphasizes **`Answer:`** and **instruction-shape** features — where the softmax contrast is centred.
- **With looser pruning**, more nodes appear under factual tokens such as **`triangle`** and **`sides`**, especially in **early and middle layers**.

**How to read the feature table.**

- **`(layer, feat)`** — index of one direction in the **Gemma 2 2B** **`gemmascope-transcoder-16k`** dictionary (Neuronpedia uses the same id).
- **Anchored token** — which input token lined up with that node in the viewer.
- **Approx. pruning** — **viewer slider** position when the screenshot was taken, **not** the JSON export knobs.
- Each cell’s prose mixes **maximum-activation corpus examples** (Top 1 % overlays on dashboards) with what lit up **on this prompt**; click through for detail.

| Nickname | `(layer, feat)` | Anchored token (≈) | Seen at viewer pruning ≈ | What fires (this prompt + dashboards; longer notes) |
|---|---:|---|---:|---|
| **Faithful △ word** | (5, [15376](https://www.neuronpedia.org/gemma-2-2b/5-gemmascope-transcoder-16k/15376)) | `triangle` | **0.80** | **On this prompt:** the node sits on **`triangle`** and reads as a clear lexical hit.<br><br>**Across the corpus:** Neuronpedia-style maximum-activating snippets keep returning the *triangle* word family—*triangle, triangles, triangular, equilateral*, plus occasional non-English variants.<br><br>**How to read it:** this is one of the most direct “shape-name” channels we saw while clicking around. Because it lives in an **early** layer, treat it as **word-level geometry context**, not as evidence the model has already encoded the full triangle inequality as a separate theorem object. |
| **“Three-ish” Trojan horse** | (7, [11752](https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/11752)) | `triangle` | **0.80** | **On this prompt:** the inputs show a **strong pull from the `triangle` embedding**, and the activation is substantial (on the order of ~11 in the clip we saved).<br><br>**Across the corpus:** the same direction’s top examples often highlight **“three” / “third”** style usage more than pure Euclidean diagrams.<br><br>**How to read it:** a useful warning that **local context** and **global statistics** can disagree. The feature can look “geometric” at one token while its cross-dataset profile is **number- or ordinality-flavored**—so always cross-check dashboards before naming the concept. |
| **Trig cheat-sheet** | (7, [12658](https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/12658)) | `the` (**after** `triangle`) | **0.80** | **On this prompt:** the highlighted node aligned with **`the` right after `triangle`**, not on `triangle` itself.<br><br>**Across the corpus:** overlays read like a mini trigonometry phrasebook—**hypotenuse, angles, sides, Pythagorean**, and similar school-math wording.<br><br>**How to read it:** important **positional lesson.** Geometry-flavored features are not guaranteed to sit on the flashiest English keyword. When you scan only the `triangle` column you can miss neighbors that still carry structured spatial language. |
| **Shape cafeteria** | (7, [9008](https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/9008)) | `triangle` | **0.80** | **On this prompt:** activity clusters with **`triangle`**, but the direction is not exclusive to that token.<br><br>**Across the corpus:** activations spread across many **figure words**—rectangles, hexagons, pentagons, circles, generic “shaped” adjectives—often in the same snippet as *triangles*.<br><br>**How to read it:** think **broad polygon / figure context** rather than a scalpel for the triangle inequality. The logit panel in one clip also **pushed against “square” tokens**, which matches a “competing family of shapes” vibe more than a single-shape expert. |
| **Sides & vertices barista** | (8, [7823](https://www.neuronpedia.org/gemma-2-2b/8-gemmascope-transcoder-16k/7823)) | `sides` | **0.80** | **On this prompt:** the node lies on **`sides`**, yet its contributors still echo **`triangle`** via embeddings and earlier-layer paths.<br><br>**Across the corpus:** examples emphasize **parts of figures**—vertices, angles between sides, hypotenuse, legs—language about how pieces of a triangle connect.<br><br>**How to read it:** this is one step closer to **relational** geometry than a bare shape label. Inequalities are stated as relations among side lengths, so directions that light up on “sides + triangle” are **more plausibly on-path** to the math content than a random format token—still not a guarantee without ablations. |
| **Protractor vibes** | (13, [15038](https://www.neuronpedia.org/gemma-2-2b/13-gemmascope-transcoder-16k/15038)) | **`is`** | **0.80** | **On this prompt:** the viewer placed the node on the predicate token **`is`**, while several listed inputs reference **`sides`** and residual error terms around that region.<br><br>**Across the corpus:** top snippets skew toward **orientation language**—angles, rotation, motion “with respect to a line,” direction in space—more than repeated glyph-level *triangle* tokens.<br><br>**How to read it:** a mid-layer feature that looks **grammatical** on the surface can still be **carrying geometric structure** underneath. Treat it as **spatial/kinematic scaffolding** that coexists with the English copula, not as a mis-click by default. |
| **Poster-child mid/high △ proof** | (16, [10626](https://www.neuronpedia.org/gemma-2-2b/16-gemmascope-transcoder-16k/10626)) | **`is`** | **0.80** | **On this prompt:** the activation was **very large compared with nearby nodes** in the same screenshot (on the order of ~49).<br><br>**Across the corpus:** overlays mix classic triangle vocabulary (**triangle, hypotenuse, angle, side length**) with **predicate-style wording** reminiscent of inequalities (*is greater than*, comparative structure).<br><br>**How to read it:** among the inspected clips, this is the **strongest qualitative bridge** between **content-heavy geometry language** and the **truth-judgment spine** ending at `Answer:`. Follow outgoing edges toward late layers **if your goal is to connect facts to logits**—but keep Section **5d**-style ablations in mind before calling the link causal. |
| **Embedding on-ramp** | *(pattern, not single id)* | `triangle` ↔ `third` corridor | **0.76–0.80** | **Pattern, not one feature id:** many separate nodes share the same qualitative picture.<br><br>**On this graph:** after easing pruning, a **bundle of paths** leaves the **`triangle` embedding** with large positive weights, then feeds **layers ~6–8**, and later splays toward tokens like **`greater`**, **`than`**, and the **`Answer:`** tail.<br><br>**How to read it:** this explains why “relax the viewer” so often **floods the bottom of the plot** before it adds mid-height nodes. Early residual traffic acts like a highway; later layers multiplex it into finer decisions. |
| **Instruction tail sirens** | *(cluster)* | `:`, **`Answer:`** | strict → loose | **Cluster-level observation:** several high-influence directions under delimiters (`:`) or **`Answer:`** showed up prominently in earlier **rank-only** summaries for the same contrast target.<br><br>**Together with screenshots:** geometric scouts can coexist **in the same graph** as **template / verdict** features—both are real, not mutually exclusive.<br><br>**How to read it:** disentangling “format head” from “content head” needs intentional comparisons. Use **Section 5c** (true vs false wording) and **Section 5d** (interventions) rather than eyeballing one static frame. |
| **Floors vs attic** | *(structural takeaway)* | all tokens | same sweep | **Structural takeaway from the same browsing session:** dragging the slider looser mostly **adds nodes in low and lower-mid layers** (roughly **L1–L9**), producing a dense “carpet” under many tokens.<br><br>**Higher layers** still appear **sparser**, especially if you only stare at the column under `triangle`.<br><br>**How to read it:** absence of a dramatic new node on `triangle` at **L20** does **not** prove the model forgot geometry upstairs—information may be **mixed and re-bottled** into directions anchored on other tokens. Scan a few columns to the right (`is`, `greater`, `Answer:`) before drawing architectural conclusions. |
| **Twin prompts sanity** | *(UI note)* | `Answer:` vs `Answer:` + tail | **0.80** | **UI check:** some captures used a clean ending **`Answer:`**; others appended stray tokens (**`False True`**) after it.<br><br>**Observation:** the **same low-layer geometry scaffolding** tended to remain visible across those variants.<br><br>**How to read it:** harmless junk at the tail should not dominate your mental model unless it changes softmax mass. Here it mostly reassures that the **triangle/sides cluster** reflects **steady internal structure**, not a single accidental screenshot framing. |

**Takeaway.** Easing pruning in the viewer mostly **reveals more early-layer dictionary directions** touching triangles, polygon language, sides, or trigonometric phrasing—the parts that dipped below stringent cuts—not a blanket proof that wholly separate “triangle inequality neurons” occupy the tallest layers unchanged.
Follow **Sections 5b–5d** (threshold sweep, contrastive rerun, intervention) whenever you move from exploratory inspection to checks with clearer causal footing.


### 5b — Node count vs. pruning threshold

This cell sweeps `node_threshold` from 0.5 to 0.99 and prints how many nodes survive at each
level. Use it to find the sweet spot: too low → unreadable graph; too high → interesting nodes
start dropping out.


In [20]:
from circuit_tracer.graph import prune_graph
import torch

graph_tf.to('cpu')
print(f"{'threshold':>12}  {'nodes kept':>12}  {'% of total':>12}")
print('-' * 40)
for thresh in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]:
    node_mask, edge_mask, _ = prune_graph(graph_tf, node_threshold=thresh, edge_threshold=0.99)
    kept = node_mask.sum().item()
    total = node_mask.numel()
    print(f"{thresh:>12.2f}  {kept:>12d}  {100*kept/total:>11.1f}%")


   threshold    nodes kept    % of total
----------------------------------------
        0.50           320          3.6%
        0.60           706          8.0%
        0.70          1389         15.8%
        0.80          2552         29.0%
        0.90          4575         52.1%
        0.95          6200         70.6%
        0.99          8083         92.0%


**What you just measured.** Each row applies `prune_graph` to the **same** attribution object `graph_tf` (same prompt, same `True − False` contrast), varying only `node_threshold` (here with `edge_threshold=0.99`). **Nodes kept** is how many graph nodes still pass the cut at that setting. **Higher** `node_threshold` ⇒ **keep more** marginal influencers (richer, busier picture); **lower** ⇒ **stricter** trimming (sparser, easier to read).

**How to read `% of total`.** That column is the fraction of **all node slots in the pruned mask** that remain visible—not “% of the whole model,” but “what fraction of *this* graph’s node grid survived.” Even at the loosest row you are still looking at the subgraph the attribution run actually built (bounded by active features, `max_feature_nodes`, and related settings), not every hypothetical location in the network.

**Three lessons from a typical printout like the one above.**

1. **Influence has a long tail.** Counts climb steeply in the middle of the table: for example, going from **~0.70** to **~0.90** can take you from **well under a thousand** nodes to **several thousand** without changing the underlying forward pass at all. Many directions carry **small** multi-hop influence; they disappear under aggressive pruning and reappear when you relax it.

2. **Figures are threshold-dependent.** If a qualitative story (“I only see geometry under `triangle` once I loosen pruning”) depends on **borderline** nodes, that story is **entangled with this knob**. In write-ups, always **name the threshold** (or show a small version of this table) whenever you show a pruned subgraph—otherwise readers cannot tell whether a node is “important” or merely “above an unstated cutoff.”

3. **Cross-link to Section 5a.** The heavy export there used **`node_threshold = 0.6`**, which in a run like the printed one keeps on the order of **~700** nodes (**single-digit percent** of the grid)—a deliberate choice for a compact JSON. A **0.8** export from the **same** `graph_tf` would keep on the order of **~2.5k** nodes (**~30%** in the same run): same science, different **field of view**.

**Bottom line.** This sweep is **calibration, not a second experiment**: it turns an invisible hyperparameter into a curve you can cite when you defend why a thesis figure is drawn at one sparsity level rather than another. Next, **Section 5c** changes the *prompt* while keeping the contrast target fixed, which answers a different question—whether features track **wording of the claim**, not just **how many** nodes a threshold leaves in view.


### 5c — Contrastive attribution (true claim vs. false claim)

**Key idea:** run the same attribution pipeline on a *false* version of the statement —
> *"Statement: For any triangle, the sum of any two sides is less than the third side. Answer:"*

— and compare the top features side by side.

- Features that appear in **both** graphs likely reflect *task format* (the `Statement:`/`Answer:`
  template, the `:` delimiter), not mathematical content.
- Features that appear **only in the true-claim graph** are candidates for content-sensitive
  circuit paths — the model is doing something different when the claim is correct.
- Features that *flip sign* (negative attribution) between the two graphs are also interesting:
  they may actively suppress `True` when the claim reverses.

**Before attribution**, the next code cell prints the **top-15 softmax next-token** mass for
*each* prompt so you can compare how the model **leans** toward ` True` / ` False` (and nearby
candidates) when only the inequality wording changes.

The **following** code cell then condenses the same forward pass into a **verdict-only table**:
`P(True)`, `P(False)`, and the **`True − False` logit gap** for the **original** prompt versus the
**negated-claim** prompt—useful to read next to the later attribution comparison.

The cell after that runs attribution and displays the two top-10 tables side by side using
`display_top_features_comparison`.


In [21]:
import torch

# False-claim wording (must match the attribution cell below).
PROMPT_FALSE_CLAIM = (
    "Statement: For any triangle, the sum of any two sides is less than the third side. Answer:"
)

idx_true = tokenizer.encode(TOKEN_TRUE, add_special_tokens=False)[-1]
idx_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]


def _print_top15_next_token(title: str, prompt_text: str, k: int = 15) -> None:
    """Softmax over last-position logits; print verdict mass and top-k tokens."""
    input_ids = model.ensure_tokenized(prompt_text)
    with torch.no_grad():
        logits, _ = model.get_activations(input_ids)
    last = logits.squeeze(0)[-1].float()
    probs = torch.softmax(last, dim=-1)
    p_t = probs[idx_true].item()
    p_f = probs[idx_false].item()
    gap_logit = (last[idx_true] - last[idx_false]).item()
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)
    clip = prompt_text if len(prompt_text) <= 120 else prompt_text[:117] + "..."
    print(clip)
    print(
        f"\nVerdict slice:  P({TOKEN_TRUE!r}) = {p_t:8.4%}    P({TOKEN_FALSE!r}) = {p_f:8.4%}    "
        f"logit(True) − logit(False) = {gap_logit:+.4f}"
    )
    topk = torch.topk(probs, k=k)
    print(f"\nTop-{k} next-token softmax (last position)")
    print(f"{'Rank':<6} {'Token':<22} {'Prob':>12} {'Logit':>10}   note")
    print("-" * 62)
    for rank, (tid, pr) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), start=1):
        tok = tokenizer.decode([tid])
        logv = last[tid].item()
        note = ""
        if tid == idx_true:
            note = "  ◄ verdict True"
        elif tid == idx_false:
            note = "  ◄ verdict False"
        print(f"{rank:<6} {repr(tok):<22} {pr:>11.3%} {logv:>10.4f}{note}")


_print_top15_next_token("TRUE claim (matches PROMPT)", PROMPT)
_print_top15_next_token("FALSE claim (incorrect inequality)", PROMPT_FALSE_CLAIM)

print("\n" + "=" * 72)
print("Lean summary: compare P(True), P(False), and logit gap between the two blocks above.")
print("=" * 72)



TRUE claim (matches PROMPT)
Statement: For any triangle, the sum of any two sides is greater than the third side. Answer:

Verdict slice:  P(' True') = 13.9406%    P(' False') =  3.1106%    logit(True) − logit(False) = +1.5000

Top-15 next-token softmax (last position)
Rank   Token                          Prob      Logit   note
--------------------------------------------------------------
1      ' True'                    13.941%    25.6250  ◄ verdict True
2      '\n'                        8.455%    25.1250
3      ' A'                        5.811%    24.7500
4      ' ('                        5.811%    24.7500
5      ' The'                      4.526%    24.5000
6      '\n\n'                      4.526%    24.5000
7      ' If'                       4.526%    24.5000
8      ' '                         3.111%    24.1250
9      ' False'                    3.111%    24.1250  ◄ verdict False
10     ' a'                        2.745%    24.0000
11     ' Statement'                2.423% 

In [25]:
import torch

# Same false-claim string as the cell above (fallback if that cell was skipped).
try:
    _ = PROMPT_FALSE_CLAIM
except NameError:
    PROMPT_FALSE_CLAIM = (
        "Statement: For any triangle, the sum of any two sides is less than the third side. Answer:"
    )

idx_true = tokenizer.encode(TOKEN_TRUE, add_special_tokens=False)[-1]
idx_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]


def _verdict_softmax_stats(prompt_text: str) -> tuple[float, float, float, float]:
    """Return (P(True), P(False), logit(True), logit(False)) at last position."""
    input_ids = model.ensure_tokenized(prompt_text)
    with torch.no_grad():
        logits, _ = model.get_activations(input_ids)
    last = logits.squeeze(0)[-1].float()
    probs = torch.softmax(last, dim=-1)
    lt = last[idx_true].item()
    lf = last[idx_false].item()
    return probs[idx_true].item(), probs[idx_false].item(), lt, lf


p_t_orig, p_f_orig, lt_orig, lf_orig = _verdict_softmax_stats(PROMPT)
p_t_neg, p_f_neg, lt_neg, lf_neg = _verdict_softmax_stats(PROMPT_FALSE_CLAIM)

gap_orig = lt_orig - lf_orig
gap_neg = lt_neg - lf_neg

print("\n" + "=" * 72)
print("5c — Verdict-token softmax (original vs negated claim, last position)")
print("=" * 72)
print(f"{'':22}  {'P(True)':>12}  {'P(False)':>12}  {'logit(T)−logit(F)':>18}")
print("-" * 72)
print(f"{'Original (true claim)':22}  {p_t_orig:>11.4%}  {p_f_orig:>11.4%}  {gap_orig:>+18.4f}")
print(f"{'Negated (false claim)':22}  {p_t_neg:>11.4%}  {p_f_neg:>11.4%}  {gap_neg:>+18.4f}")
print("-" * 72)
print(
    f"{'Δ (negated − original)':22}  "
    f"{(p_t_neg - p_t_orig):>+11.4%}  {(p_f_neg - p_f_orig):>+11.4%}  {(gap_neg - gap_orig):>+18.4f}"
)



5c — Verdict-token softmax (original vs negated claim, last position)
                             P(True)      P(False)   logit(T)−logit(F)
------------------------------------------------------------------------
Original (true claim)      13.9406%      3.1106%             +1.5000
Negated (false claim)      13.7533%      3.9404%             +1.2500
------------------------------------------------------------------------
Δ (negated − original)     -0.1873%     +0.8298%             -0.2500


In [22]:
from circuit_tracer import attribute
from circuit_tracer.utils.demo_utils import display_top_features_comparison, get_top_features, cleanup_cuda

# PROMPT_FALSE_CLAIM is defined in the previous cell (softmax comparison).

print("Running attribution on the FALSE-claim prompt…")
graph_false = attribute(
    PROMPT_FALSE_CLAIM,
    model,
    attribution_targets=[TOKEN_TRUE, TOKEN_FALSE],
    **ATTR_KWARGS,
)

features_true_claim, scores_true_claim = get_top_features(graph_tf, n=10)
features_false_claim, scores_false_claim = get_top_features(graph_false, n=10)

print("\nTrue-claim prompt:", PROMPT[:80])
print("False-claim prompt:", PROMPT_FALSE_CLAIM[:80])

display_top_features_comparison(
    feature_sets={
        "True claim (correct)": features_true_claim,
        "False claim (incorrect)": features_false_claim,
    },
    scores_sets={
        "True claim (correct)": scores_true_claim,
        "False claim (incorrect)": scores_false_claim,
    },
    neuronpedia_model=np_model,
    neuronpedia_set=np_set,
)

# Which features are shared?
set_true  = set(map(tuple, features_true_claim))
set_false = set(map(tuple, features_false_claim))
shared = set_true & set_false
print(f"\nShared top-10 features: {len(shared)}")
for f in sorted(shared):
    print(f"  {f}")

cleanup_cuda()


Phase 0: Precomputing activations and vectors


Running attribution on the FALSE-claim prompt…


Precomputation completed in 1.86s
Found 23539 active features
Phase 1: Running forward pass
Forward pass completed in 0.43s
Phase 2: Building input vectors
Using 2 specified logit targets with cumulative probability 0.1768
Will include 8192 of 23539 feature nodes
Input vectors built in 5.52s
Phase 3: Computing logit attributions
Logit attributions completed in 0.29s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [00:09<00:00, 870.26it/s]
Feature attributions completed in 9.42s
Attribution completed in 20.83s



True-claim prompt: Statement: For any triangle, the sum of any two sides is greater than the third 
False-claim prompt: Statement: For any triangle, the sum of any two sides is less than the third sid


#,Node,Score
1,"(16, 21, 14143)",0.0028
2,"(20, 21, 12997)",0.0026
3,"(24, 21, 4640)",0.0025
4,"(18, 21, 15770)",0.0022
5,"(17, 21, 6448)",0.0016
6,"(0, 2, 16200)",0.0016
7,"(18, 21, 2940)",0.0015
8,"(19, 21, 6538)",0.0015
9,"(21, 21, 7787)",0.0012
10,"(17, 21, 15506)",0.0012



Shared top-10 features: 9
  (0, 2, 16200)
  (16, 21, 14143)
  (17, 21, 6448)
  (18, 21, 2940)
  (18, 21, 15770)
  (19, 21, 6538)
  (20, 21, 12997)
  (21, 21, 7787)
  (24, 21, 4640)


### 5e  -  When the same feature pushes "True vs False" the *other* way on a false claim

We run attribution twice: once on the **true** triangle-inequality prompt, once on a **false** wording (same shape, but the inequality is flipped). For each internal **feature** that appears in **both** graphs, we ask: "Does this feature nudge the model toward **True** or toward **False**?" - summarized as one **signed** number per prompt.

**What the printed table is for.** It keeps only features where that signed nudge **points one way on the true sentence and the opposite way on the false sentence**, and only on token positions **5–18** (the body of the claim, not the `Statement:` / `Answer:` framing). Rows are sorted so the **largest change** between the two prompts appears first. That is a simple way to surface machinery that **reacts to the claim being wrong**, not a list of "the most important neurons in the whole model."

**How to read one line.** `(layer, pos, feature)` is *where* the feature lives: layer depth, which **input token** (`pos`), and which dictionary direction (`feature`). `s_orig` and `s_neg` are the two signed nudges; `Δ` is how much the false wording **moves** that nudge. **`***`** means `pos` is exactly the token where **"greater"** (true prompt) and **"less"** (false prompt) line up in the tokenizer - so those rows are especially tied to **how the inequality is worded**.

**Neuronpedia** links use `(layer, feature)` so you can see real firing examples on other text - not just this prompt.

The subsection **Section 5e: top-10 sign flips, read next to Neuronpedia** below combines the printed table with each row's Neuronpedia auto-labels (fetched from the public pages).


In [ ]:
import torch
from circuit_tracer.graph import compute_node_influence

POS_MIN, POS_MAX = 5, 18
MAX_PRINT_ROWS = 60  # cap long tables; raise if you want the full intersection printed


def _decoded_tokens(prompt: str) -> list[str]:
    ids = model.ensure_tokenized(prompt)
    return [tokenizer.decode([int(tid)]) for tid in ids.view(-1).tolist()]


def _first_pos_containing(decoded: list[str], needle: str) -> int | None:
    for i, piece in enumerate(decoded):
        if needle in piece:
            return i
    return None


def _cmp_token_positions() -> tuple[int | None, int | None, bool]:
    """Indices of `greater` / `less` and whether they align across prompts."""
    d0 = _decoded_tokens(PROMPT)
    d1 = _decoded_tokens(PROMPT_FALSE_CLAIM)
    pg = _first_pos_containing(d0, "greater")
    pl = _first_pos_containing(d1, "less")
    aligned = pg is not None and pl is not None and pg == pl and len(d0) == len(d1)
    return pg, pl, aligned


def _signed_true_minus_false_vector(graph) -> torch.Tensor:
    """Weights +1 on first logit target, −1 on second (True then False from this notebook)."""
    n_logits = len(graph.logit_targets)
    if n_logits != 2:
        raise ValueError(f"Expected exactly 2 logit targets, got {n_logits}")
    n_nodes = graph.adjacency_matrix.shape[0]
    device = graph.adjacency_matrix.device
    w = torch.zeros(n_nodes, dtype=torch.float32, device=device)
    w[-n_logits] = 1.0
    w[-n_logits + 1] = -1.0
    return w


def _per_selected_signed_scores(graph) -> torch.Tensor:
    """Shape (n_selected,) — same node order as graph.selected_features."""
    w = _signed_true_minus_false_vector(graph)
    node_inf = compute_node_influence(graph.adjacency_matrix, w)
    n_feat = len(graph.selected_features)
    return node_inf[:n_feat]


def _scores_by_triple(graph) -> dict[tuple[int, int, int], float]:
    af = graph.active_features
    sel = graph.selected_features
    vec = _per_selected_signed_scores(graph)
    out: dict[tuple[int, int, int], float] = {}
    for i in range(len(sel)):
        L, P, F = af[sel[i]].tolist()
        out[(int(L), int(P), int(F))] = float(vec[i].item())
    return out


np_model = globals().get("np_model") or "gemma-2-2b"
np_set = globals().get("np_set") or "gemmascope-transcoder-16k"


def _neuronpedia_url(layer: int, feat: int) -> str:
    return f"https://www.neuronpedia.org/{np_model}/{layer}-{np_set}/{feat}"


# Graph.to mutates in place — move both graphs to CPU for cheap repeated reads.
graph_tf.to("cpu")
graph_false.to("cpu")

scores_orig = _scores_by_triple(graph_tf)
scores_neg = _scores_by_triple(graph_false)

keys_orig = set(scores_orig)
keys_neg = set(scores_neg)
both = keys_orig & keys_neg

pos_greater, pos_less, cmp_aligned = _cmp_token_positions()
print("\n=== 5e — token alignment (greater / less) ===")
print(f"  pos('greater') in PROMPT:              {pos_greater}")
print(f"  pos('less')   in PROMPT_FALSE_CLAIM: {pos_less}")
print(f"  same index & equal length:           {cmp_aligned}")
if not cmp_aligned:
    print(
        "  [warn] Prompts are not perfectly aligned — `(layer,pos,feat)` still keys the same index, "
        "but surface text under `pos` may differ."
    )

candidates: list[tuple[int, int, int, float, float, float, float]] = []
for triple in both:
    L, P, F = triple
    if not (POS_MIN <= P <= POS_MAX):
        continue
    s0 = scores_orig[triple]
    s1 = scores_neg[triple]
    if s0 * s1 >= 0:
        continue
    delta = s1 - s0
    mag = abs(delta)
    candidates.append((L, P, F, s0, s1, delta, mag))

candidates.sort(key=lambda r: -r[-1])
rows_5e = list(candidates)

n_both = len(both)
n_pos = sum(1 for (L, P, F) in both if POS_MIN <= P <= POS_MAX)
n_flip = len(candidates)
print("\n=== 5e — intersection → opposite signs → pos band ===")
print(f"  selected triples in ORIG: {len(keys_orig):6d}")
print(f"  selected triples in NEG:  {len(keys_neg):6d}")
print(f"  intersection (both):     {n_both:6d}")
print(f"  after pos in [{POS_MIN},{POS_MAX}]: {n_pos:6d}")
print(f"  after opposite signs:    {n_flip:6d}")

if not candidates:
    print(
        "\n[5e] No rows after filters. Try raising max_feature_nodes, or relax opposite-sign "
        "criterion (e.g. include near-zero)."
    )
else:
    shown = candidates[:MAX_PRINT_ROWS]
    hdr = (
        f"{'#':<4} {'(L,P,F)':<16} {'cmp':>4} "
        f"{'s_orig':>12} {'s_neg':>12} {'Δ=neg−orig':>14} {'|Δ|':>12}  Neuronpedia"
    )
    print("\n" + hdr)
    print("-" * (len(hdr) + 8))
    for i, (L, P, F, s0, s1, delta, mag) in enumerate(shown, start=1):
        cmp_mark = ""
        if cmp_aligned and pos_greater is not None and P == pos_greater == pos_less:
            cmp_mark = "***"
        url = _neuronpedia_url(L, F)
        print(
            f"{i:<4} ({L:>2},{P:>2},{F:>5}) {cmp_mark:>3} "
            f"{s0:>+12.5e} {s1:>+12.5e} {delta:>+14.5e} {mag:>12.5e}  {url}"
        )
    if len(candidates) > MAX_PRINT_ROWS:
        print(f"\n… truncated: showing {MAX_PRINT_ROWS} of {len(candidates)} rows (edit MAX_PRINT_ROWS).")

print(
    "\nLegend: cmp `***` = token position matches the shared greater/less slot when prompts align."
)
print("Interpretation: opposite signs are a *screen* for claim-sensitive hooks, not proof of correctness.")



=== 5e — token alignment (greater / less) ===
  pos('greater') in PROMPT:              14
  pos('less')   in PROMPT_FALSE_CLAIM: 14
  same index & equal length:           True

=== 5e — intersection → opposite signs → pos band ===
  selected triples in ORIG:   8192
  selected triples in NEG:    8192
  intersection (both):       7654
  after pos in [5,18]:   3129
  after opposite signs:       138

#    (L,P,F)           cmp       s_orig        s_neg     Δ=neg−orig          |Δ|  Neuronpedia
-----------------------------------------------------------------------------------------------------
1    ( 6,14, 4963) *** -6.86512e-06 +1.01245e-04   +1.08110e-04  1.08110e-04  https://www.neuronpedia.org/gemma-2-2b/6-gemmascope-transcoder-16k/4963
2    ( 1,14,12924) *** -3.60783e-05 +2.27872e-05   +5.88655e-05  5.88655e-05  https://www.neuronpedia.org/gemma-2-2b/1-gemmascope-transcoder-16k/12924
3    (10,13, 3694)     +3.64130e-05 -1.17248e-05   -4.81378e-05  4.81378e-05  https://www.neuronpedia.

#### Interpretation

The blurbs below are taken from each feature's **Neuronpedia** page. They describe **typical** firing patterns on broad text, **not** a guarantee about this triangle prompt. The **numbers** in parentheses are from the notebook's printed **5e** table: signed influence on `logit(True) − logit(False)` for the **true** claim (`s_orig`) vs the **false** claim (`s_neg`), with `***` when `pos` matches the aligned **greater / less** token (here, **14** on both prompts).

---

**1.** `(6, 14, 4963)` `***` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/6-gemmascope-transcoder-16k/4963) — Table: small **negative** `s_orig`, large **positive** `s_neg` (largest `|Δ|` in the top ten). Neuronpedia: *"words related to mathematical logic and proofs, including references"* / *"strict"*. **Read together:** the feature sits on the **inequality word itself** and flips from mildly pushing against the `True − False` gap to strongly pushing the other way when the claim is false. The dashboard label reads like **proof-style and formal-math language**; that is compatible with treating this direction as part of a **logical / comparative scaffold** around the predicate, not as evidence the model ran a full proof.

**2.** `(1, 14, 12924)` `***` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/1-gemmascope-transcoder-16k/12924) — Table: **negative** then **positive** on the same comparative token, large `|Δ|`. Neuronpedia: *"mathematical notation and symbols"* / *"dim"*. **Read together:** very early layer machinery on **pos 14** (greater/less) whose corpus profile is **notation-heavy**. A natural story is: token-level **symbol / shorthand detectors** are being **re-polarised** when *greater* becomes *less*, even before the network has fully unpacked the sentence.

**3.** `(10, 13, 3694)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/10-gemmascope-transcoder-16k/3694) — Table: **positive** then **negative**. Neuronpedia: *"mathematical terms or concepts"* / *"mathematical relationships"*; the page also lists **least** among strong negative logits. **Read together:** mid-high layer, still **math-shaped** in auto-interp, and the logit probe hints at **ordered comparisons** ("least" vs "greater"). The flip next to *greater*/*less* fits spreading comparative semantics across neighbouring positions, not only on the head token.

**4.** `(0, 15, 8167)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/8167) — Table: **negative** then **positive** at pos 15 (than). Neuronpedia labels: the phrase "more than" / than. **Read together:** layer 0 is essentially embedding-adjacent; the label is overtly about **comparative phrasing**. That lines up cleanly with a sentence containing **greater … than**: the feature is plausibly carrying scalar-comparison glue whose signed effect on the verdict hinge reverses when the inequality is negated.

**5.** `(1, 15, 12610)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/1-gemmascope-transcoder-16k/12610) — Table: **positive** then **negative** at pos 15 (than). Neuronpedia labels: snow and related wording / snow. **Read together:** the attribution table says this direction is claim-sensitive, but the corpus-level story is **weather**, not triangles. Treat this as a warning example: **the same (layer, pos, feat) can look important in the graph while Neuronpedia names an unrelated theme** (polysemanticity, superposition, or different triggers on different strings).

**6.** `(7, 13, 3045)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/3045) — Table: **positive** then **negative**. Neuronpedia labels: subtraction / subtracting (plus math notation) / traction, subtracting. **Read together:** another math-operation flavour, slightly syntax / arithmetic oriented. Sitting mid-network on pos 13, it plausibly tracks **relational arithmetic language** near the inequality; the sign flip says the false wording reassigns how this direction votes on `True − False`.

**7.** `(3, 15, 6861)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/3-gemmascope-transcoder-16k/6861) — Table: **negative** then **positive** at pos 15 (than). Neuronpedia: *"mathematical comparisons using words such as greater, higher, or equal"* / *"than"*. **Read together:** this is the most on-the-nose auto-label for your prompt wording. The picture is: **comparative semantics** need not be registered only on the single head token; a neighbouring position can still host a greater / higher / equal detector whose influence on the hinge flips when the claim is false.

**8.** `(1, 14, 11731)` `***` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/1-gemmascope-transcoder-16k/11731) — Table: **negative** then **positive** on pos 14 (greater/less). Neuronpedia: *"mathematical equations and symbols"*. **Read together:** another early-layer, comparative-slot direction with an **equation / symbol** gloss—overlaps in *role* with row 2 but not the same feature index. Good illustration that several dictionary directions can share one token column yet differ in corpus semantics; only Neuronpedia (or manual snippet review) separates them.

**9.** `(0, 12, 4872)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/0-gemmascope-transcoder-16k/4872) — Table: **negative** then **positive** at pos 12 (sides). Neuronpedia: *"mathematical equations and formulas"* / *"+"*. **Read together:** early layer, a few tokens left of the comparative; still **math-form** oriented. Fits the idea that some claim-sensitive signal is already present in low-level channels before mid layers mix it.

**10.** `(5, 13, 5889)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/5-gemmascope-transcoder-16k/5889) — Table: **negative** `s_orig`, **tiny positive** `s_neg`. Neuronpedia: *"words or phrases related to data, coding, and computer terminology"*. **Read together:** like row 5, the global description is far from Euclidean geometry, yet the feature passes the opposite-sign screen. Use this row to temper rhetoric: the filter is algebraic, not "semantic purity."

---

#### Pulling the ten rows together

**Comparative language:** Rows **4** (*more than* / *than*), **7** (*greater, higher, equal* / *than*), and **3** (math relationships; *least* in logits) all play into the model’s understanding of comparisons—specifically, the contrast between "greater" and "less." Complementing this, rows **1**, **2**, and **8** bring in elements of *logic, proof, notation,* and *equation*, often centered around or near the *comparative token*.

**Polysemantic outliers:** Rows **5** (focused on *snow*) and **10** (linked to *code/data* jargon) serve as clear reminders that just because a direction is sensitive to the claim does not mean it maps neatly onto a single, intuitive human theme. For this, cross-checking with Neuronpedia is invaluable.

**Layer-wise structure:** Many of these flips show up in *early layers (0–1)*—rows 2, 4, 5, 8, and 9—highlighting the role of early comparative and notation detectors, which are sensitive to the true versus false prompt. At the same time, *middle layers* (3, 6, 7, 10) lend support with more math-flavored or generic representations, blending these signals as the network proceeds.

---

#### What this combined view does *not* prove

Neuronpedia labels are **noisy summaries** of a pretraining corpus, not ground truth labels for each feature. The table shows sensitivity of an attribution-derived score to one contrastive rewrite. Neither source alone establishes necessity, causality, or geometric understanding; interventions and more prompts still matter.




### 5f — Geometry gallery

**Goal:** Start from the hand-checked §5a gallery—each entry has a nickname, layer, and feat. For each entry, resolve its pos within the PROMPT using viewer-style heuristics:
   - "triangle", "the" after "triangle", "sides", or the copula before "greater".
Retain only those with pos in the range [5, 18] (drop "Statement:" and "Answer:").

For every resulting (layer, pos, feat) triple, look up its signed `True − False` influence on
both the original statement and the negated one. Rows are sorted by `|s_neg − s_orig|` so that the most strongly claim-sensitive moves appear first. An extra *flip?* column highlights when the sign flips between graphs.

**After you run the code cell:** open **5f in plain English** (next markdown cell) for skipped rows, the printed table, and short Neuronpedia takeaways.

In [31]:
# §5f — §5a gallery: resolve (layer, feat) → pos on PROMPT (viewer anchors), then signed
# True−False influence on both graphs. No opposite-sign filter; flip? is informational.
import torch
from circuit_tracer.graph import compute_node_influence

POS_MIN, POS_MAX = 5, 18

# (nickname, layer, feat, anchor_kind) — anchor_kind matches the §5a "Anchored token" column.
GALLERY_5F: list[tuple[str, int, int, str]] = [
    ("Faithful △ word", 5, 15376, "triangle"),
    ("Three-ish Trojan horse", 7, 11752, "triangle"),
    ("Trig cheat-sheet", 7, 12658, "the_after_triangle"),
    ("Shape cafeteria", 7, 9008, "triangle"),
    ("Sides & vertices barista", 8, 7823, "sides"),
    ("Protractor vibes", 13, 15038, "copula_before_greater"),
    ("Poster-child mid/high △ proof", 16, 10626, "copula_before_greater"),
]


def _decoded_tokens_5f(prompt: str) -> list[str]:
    ids = model.ensure_tokenized(prompt)
    return [tokenizer.decode([int(tid)]) for tid in ids.view(-1).tolist()]


def _first_pos_containing_5f(decoded: list[str], needle: str, start: int = 0) -> int | None:
    for i in range(start, len(decoded)):
        if needle in decoded[i]:
            return i
    return None


def _resolve_gallery_pos(decoded: list[str], anchor_kind: str) -> int | None:
    if anchor_kind == "triangle":
        return _first_pos_containing_5f(decoded, "triangle")
    if anchor_kind == "the_after_triangle":
        t = _first_pos_containing_5f(decoded, "triangle")
        if t is None:
            return None
        return _first_pos_containing_5f(decoded, "the", start=t + 1)
    if anchor_kind == "sides":
        return _first_pos_containing_5f(decoded, "sides")
    if anchor_kind == "copula_before_greater":
        g = _first_pos_containing_5f(decoded, "greater")
        if g is None:
            return None
        for k in range(g - 1, -1, -1):
            core = (
                decoded[k]
                .replace("▁", " ")
                .strip()
                .strip(".,;:'\"")
                .lower()
            )
            if core == "is":
                return k
        return None
    raise ValueError(f"unknown anchor_kind {anchor_kind!r}")


def _signed_true_minus_false_vector(graph) -> torch.Tensor:
    n_logits = len(graph.logit_targets)
    if n_logits != 2:
        raise ValueError(f"Expected exactly 2 logit targets, got {n_logits}")
    n_nodes = graph.adjacency_matrix.shape[0]
    device = graph.adjacency_matrix.device
    w = torch.zeros(n_nodes, dtype=torch.float32, device=device)
    w[-n_logits] = 1.0
    w[-n_logits + 1] = -1.0
    return w


def _per_selected_signed_scores(graph) -> torch.Tensor:
    w = _signed_true_minus_false_vector(graph)
    node_inf = compute_node_influence(graph.adjacency_matrix, w)
    n_feat = len(graph.selected_features)
    return node_inf[:n_feat]


def _scores_by_triple_5f(graph) -> dict[tuple[int, int, int], float]:
    af = graph.active_features
    sel = graph.selected_features
    vec = _per_selected_signed_scores(graph)
    out: dict[tuple[int, int, int], float] = {}
    for i in range(len(sel)):
        L, P, F = af[sel[i]].tolist()
        out[(int(L), int(P), int(F))] = float(vec[i].item())
    return out


def _ensure_scores_5f():
    """Reuse §5e globals when present and non-empty; else build from graphs (CPU)."""
    so, sn = globals().get("scores_orig"), globals().get("scores_neg")
    if isinstance(so, dict) and isinstance(sn, dict) and len(so) > 0 and len(sn) > 0:
        return so, sn
    graph_tf.to("cpu")
    graph_false.to("cpu")
    return _scores_by_triple_5f(graph_tf), _scores_by_triple_5f(graph_false)


np_model = globals().get("np_model") or "gemma-2-2b"
np_set = globals().get("np_set") or "gemmascope-transcoder-16k"


def _neuronpedia_url_5f(layer: int, feat: int) -> str:
    if "_neuronpedia_url" in dir():
        return _neuronpedia_url(layer, feat)
    return f"https://www.neuronpedia.org/{np_model}/{layer}-{np_set}/{feat}"


def _cmp_token_positions_5f() -> tuple[int | None, int | None, bool]:
    if "_cmp_token_positions" in dir():
        return _cmp_token_positions()
    d0 = _decoded_tokens_5f(PROMPT)
    d1 = _decoded_tokens_5f(PROMPT_FALSE_CLAIM)
    pg = _first_pos_containing_5f(d0, "greater")
    pl = _first_pos_containing_5f(d1, "less")
    aligned = pg is not None and pl is not None and pg == pl and len(d0) == len(d1)
    return pg, pl, aligned


scores_orig, scores_neg = _ensure_scores_5f()
both = set(scores_orig) & set(scores_neg)

d_prompt = _decoded_tokens_5f(PROMPT)
pos_greater, pos_less, cmp_aligned = _cmp_token_positions_5f()

rows_5f: list[tuple[str, int, int, int, float, float, float, float, bool]] = []
for nickname, L, F, akind in GALLERY_5F:
    P = _resolve_gallery_pos(d_prompt, akind)
    if P is None:
        print(f"[5f] {nickname!r}: could not resolve pos ({akind!r}) on PROMPT — skipped.")
        continue
    if not (POS_MIN <= P <= POS_MAX):
        print(f"[5f] {nickname!r}: pos={P} outside [{POS_MIN},{POS_MAX}] — skipped.")
        continue
    triple = (L, P, F)
    if triple not in both:
        print(
            f"[5f] {nickname!r}: triple {triple} not in intersection of selected subgraphs — skipped."
        )
        continue
    s0 = scores_orig[triple]
    s1 = scores_neg[triple]
    delta = s1 - s0
    mag = abs(delta)
    flip = s0 * s1 < 0
    rows_5f.append((nickname, L, P, F, s0, s1, delta, mag, flip))

rows_5f.sort(key=lambda r: -r[7])

print("\n=== 5f — §5a gallery as (layer, pos, feat); both graphs; |Δ| rank; flip? info ===")
if not rows_5f:
    print("  (no rows — check §5c graphs, gallery pos resolution, intersection, or pos band)")
else:
    hdr = (
        f"{'#':<4} {'nickname':<32} {'(L,P,F)':<16} {'cmp':>4} "
        f"{'flip?':>6} {'s_orig':>12} {'s_neg':>12} {'|Δ|':>12}  Neuronpedia"
    )
    print(hdr)
    print("-" * (len(hdr) + 10))
    for i, (name, L, P, F, s0, s1, delta, mag, flip) in enumerate(rows_5f, start=1):
        cmp_mark = ""
        if cmp_aligned and pos_greater is not None and P == pos_greater == pos_less:
            cmp_mark = "***"
        flip_s = "Y" if flip else "n"
        url = _neuronpedia_url_5f(L, F)
        print(
            f"{i:<4} {name:<32} ({L:>2},{P:>2},{F:>5}) {cmp_mark:>3} "
            f"{flip_s:>6} {s0:>+12.5e} {s1:>+12.5e} {mag:>12.5e}  {url}"
        )

print("\n(Use these (L, P, F) triples verbatim in §5d-style interventions.)")
print(
    "Legend: flip? = opposite signs on the two prompts (§5e-style screen), not a filter here."
)


[5f] 'Protractor vibes': triple (13, 13, 15038) not in intersection of selected subgraphs — skipped.
[5f] 'Poster-child mid/high △ proof': triple (16, 13, 10626) not in intersection of selected subgraphs — skipped.

=== 5f — §5a gallery as (layer, pos, feat); both graphs; |Δ| rank; flip? info ===
#    nickname                         (L,P,F)           cmp  flip?       s_orig        s_neg          |Δ|  Neuronpedia
--------------------------------------------------------------------------------------------------------------------------------
1    Sides & vertices barista         ( 8,12, 7823)          n -3.27948e-04 -2.78172e-04  4.97768e-05  https://www.neuronpedia.org/gemma-2-2b/8-gemmascope-transcoder-16k/7823
2    Faithful △ word                  ( 5, 5,15376)          n -2.00109e-04 -2.29722e-04  2.96130e-05  https://www.neuronpedia.org/gemma-2-2b/5-gemmascope-transcoder-16k/15376
3    Three-ish Trojan horse           ( 7, 5,11752)          n -2.52347e-05 -3.07247e-05  5.49007e-06  

#### Interpretation


**Quick read of the columns**

- **`s_orig` / `s_neg`** — how this feature nudges **`logit(True) − logit(False)`** on the true vs false wording.
- **`flip?`** — `Y` only if the sign flips. In your run it is **`n` every time**: same direction of nudge on both prompts; only the **size** changes (that is what `|Δ|` ranks).
- **`|Δ|`** — how much the false wording **moves** that nudge, up or down.

---

**1. Sides and vertices barista** — `(8, 12, 7823)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/8-gemmascope-transcoder-16k/7823)

- Table: both scores are **negative**; the false claim is a bit less negative 
- Neuronpedia: **geometry** and equations / formulas.
- In one sentence: still "pushing" the hinge the same way, but weaker on the false sentence—so the claim change matters, without a sign flip.

**2. Faithful triangle word** — `(5, 5, 15376)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/5-gemmascope-transcoder-16k/15376)

- Table: both **negative**; the false claim is more*negative than the true one.
- Neuronpedia: mentions **triangle** (fits the anchor) but also scientific / biological fluff like codon on broad text.
- In one sentence: the false wording strengthens the same-signed nudge; treat the biology bit as a heads-up that one feature id can mean several things in the wild.

**3. Three-ish Trojan horse** — `(7, 5, 11752)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/11752)

- Table: both **negative**; very small `|Δ|` (barely moves).
- Neuronpedia: counting / groups of things and football; logits still show **THREE**-style tokens.
- In one sentence: good sanity check row—the gallery already warned this might track "three" more than Euclid; the scores barely budge between prompts.

**4. Trig cheat-sheet** — `(7, 7, 12658)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/12658)

- Table: both **positive**; false claim slightly more positive (.
- Neuronpedia: symbols / variables / equations and **geometry and math**.
- In one sentence: sits on *the* after *triangle*; labels read like a notation / trig-adjacent channel that amps up a little on the false wording.

**5. Shape cafeteria** — `(7, 5, 9008)` — [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/7-gemmascope-transcoder-16k/9008)

- Table: both **negative**; tiny `|Δ|` (almost flat).
- Neuronpedia: **polygons / shapes**; logits lean against square tokens.
- In one sentence: broad shape-bag feature on *triangle*; for this contrast it is nearly unchanged between prompts.

---

#### Closing points

1. **Neuronpedia mostly agrees** with the gallery for the five rows that printed (geometry, sides, triangle, notation, polygons).
2. **Do not read this block like Section 5e.** 5e hunted **opposite signs**. Here every printed row keeps the same sign; you are looking at how much the false claim rescales the same direction of push.



## Section 6 -- Interventions

We have three questions, tackled in order:

**6a -- Do the top-five attribution-ranked features control the verdict?**  
These are ranks 1--5 from `get_top_features(graph_tf)` (same order as the Section 5 feature table). For each feature we **zero-ablate** and **scale** the transcoder activation by **10x** relative to a clean baseline forward on **that** prompt (same recipe as Anthropic `demos/attribution_targets_demo.ipynb`). Output matches **6b** (per-feature tables + `display_topk`), then a **cross-feature summary**.

**6b -- Sign-flip features (Section 5e):** same ablate / **10x**-stim grid on both prompts.

**6c -- Geometry gallery (Section 5f):** same grid for the five gallery hooks.

For every cell, the output shows the `logit(True) - logit(False)` gap and top-token tables.



### 6a -- Top **five** attribution features: ablate and stimulate on both statements

The list is **ranks 1--5** from the true-claim attribution (`features_tf[:5]` after `get_top_features(graph_tf, n=10)`). If you have not run that cell, the code falls back to `get_top_features(graph_tf, n=5)`.

**Ablation:** set the transcoder activation at `(L,P,F)` to **0**.

**Stimulation (Anthropic convention):** `model.get_activations(prompt)` records baseline activations `a` at each layer/position/feature. The intervention **target** value passed to `feature_intervention` is **`STIM_MUL * a`** with **`STIM_MUL = 10.0`**, matching `10.0 * activations[layer, pos, feat_idx]` in `demos/attribution_targets_demo.ipynb`. True- and false-statement runs use **their own** baseline `a` (`a_true` vs `a_false`). If `a = 0`, stimulation is a no-op.

| | Zero-ablate | Stimulate |
|---|---|---|
| **True statement** | `0` | `10 * a_true` |
| **False statement** | `0` | `10 * a_false` |

**What to look for:** compare **T-stim** vs **T-abl** in the cross-feature summary. Large multi-hop rank does not guarantee a large gap move.



In [39]:
import torch
from functools import partial
from circuit_tracer.utils.demo_utils import (
    display_topk_token_predictions,
    get_top_features,
)

display_topk = partial(display_topk_token_predictions, tokenizer=tokenizer)

try:
    TOP5_FEATURES = [(L, P, F) for L, P, F in features_tf[:5]]
except NameError:
    TOP5_FEATURES, _ = get_top_features(graph_tf, n=5)

# Anthropic attribution_targets_demo.ipynb: 10.0 * activations[layer, pos, feat_idx]
STIM_MUL = 10.0

idx_true = tokenizer.encode(TOKEN_TRUE, add_special_tokens=False)[-1]
idx_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
key_tokens = [(TOKEN_TRUE, idx_true), (TOKEN_FALSE, idx_false)]


def _gap(logits):
    last = logits.squeeze(0)[-1].float()
    return (last[idx_true] - last[idx_false]).item()


with torch.inference_mode():
    _, act_prompt_T = model.get_activations(PROMPT, sparse=False)
    _, act_prompt_F = model.get_activations(PROMPT_FALSE_CLAIM, sparse=False)
    base_true_logits, _ = model.feature_intervention(PROMPT, [])
    base_false_logits, _ = model.feature_intervention(PROMPT_FALSE_CLAIM, [])

gap_base_true = _gap(base_true_logits)
gap_base_false = _gap(base_false_logits)

summary_rows = []

for feat_idx, (L, P, F) in enumerate(TOP5_FEATURES, start=1):
    label = f"({L},{P},{F})"
    a_T = float(act_prompt_T[L, P, F].item())
    a_F = float(act_prompt_F[L, P, F].item())
    stim_T = STIM_MUL * a_T
    stim_F = STIM_MUL * a_F

    print("\n" + "=" * 60)
    print(f"Attr rank {feat_idx}/5 (by graph influence): {label}")
    print(f"  baseline activation  a_true={a_T:+.6f}  a_false={a_F:+.6f}  |  stim target ({STIM_MUL:g}x) -> {stim_T:+.6f}, {stim_F:+.6f}")
    print("=" * 60)

    with torch.inference_mode():
        abl_t_logits = model.feature_intervention(PROMPT, [(L, P, F, 0.0)])[0]
        stim_t_logits = model.feature_intervention(PROMPT, [(L, P, F, stim_T)])[0]
        abl_f_logits = model.feature_intervention(PROMPT_FALSE_CLAIM, [(L, P, F, 0.0)])[0]
        stim_f_logits = model.feature_intervention(PROMPT_FALSE_CLAIM, [(L, P, F, stim_F)])[0]

    gaps = {
        "base_T": gap_base_true,
        "abl_T": _gap(abl_t_logits),
        "stim_T": _gap(stim_t_logits),
        "base_F": gap_base_false,
        "abl_F": _gap(abl_f_logits),
        "stim_F": _gap(stim_f_logits),
    }

    for name, (lbl, base_key) in [
        ("base_T", ("True  baseline", None)),
        ("abl_T", ("True  ablate", "base_T")),
        ("stim_T", ("True  stimulate", "base_T")),
        ("base_F", ("False baseline", None)),
        ("abl_F", ("False ablate", "base_F")),
        ("stim_F", ("False stimulate", "base_F")),
    ]:
        d = f"{gaps[name] - gaps[base_key]:+.4f}" if base_key else "   --"
        print(f"  {lbl:<22}  gap={gaps[name]:>+8.4f}  delta={d}")

    summary_rows.append((label, gaps))

    print("\n  -- True: ablate vs baseline --")
    display_topk(PROMPT, base_true_logits, abl_t_logits, key_tokens=key_tokens)

    print("  -- True: stimulate vs baseline --")
    display_topk(PROMPT, base_true_logits, stim_t_logits, key_tokens=key_tokens)

    print("  -- False: ablate vs baseline --")
    display_topk(PROMPT_FALSE_CLAIM, base_false_logits, abl_f_logits, key_tokens=key_tokens)

    print("  -- False: stimulate vs baseline --")
    display_topk(PROMPT_FALSE_CLAIM, base_false_logits, stim_f_logits, key_tokens=key_tokens)

print("\n" + "=" * 78)
print("CROSS-FEATURE SUMMARY (delta = gap change vs same-prompt baseline)")
print("=" * 78)
print(f'  {"Feature":<18}  {"T-abl":>8}  {"T-stim":>8}  {"F-abl":>8}  {"F-stim":>8}')
print("-" * 60)
for lbl, g in summary_rows:
    print(
        f"  {lbl:<18}"
        f'  {g["abl_T"] - g["base_T"]:>+8.4f}'
        f'  {g["stim_T"] - g["base_T"]:>+8.4f}'
        f'  {g["abl_F"] - g["base_F"]:>+8.4f}'
        f'  {g["stim_F"] - g["base_F"]:>+8.4f}'
    )

cleanup_cuda()




Attr rank 1/5 (by graph influence): (16,21,14143)
  baseline activation  a_true=+40.250000  a_false=+40.500000  |  stim target (10x) -> +402.500000, +405.000000
  True  baseline          gap= +1.5000  delta=   --
  True  ablate            gap= +1.6250  delta=+0.1250
  True  stimulate         gap= +2.1250  delta=+0.6250
  False baseline          gap= +1.2500  delta=   --
  False ablate            gap= +1.5000  delta=+0.2500
  False stimulate         gap= +2.0000  delta=+0.7500

  -- True: ablate vs baseline --


Token,Probability,Distribution
True,0.140,14.0%
,0.084,8.4%
(,0.058,5.8%
A,0.058,5.8%
,0.045,4.5%
Token,Probability,Distribution
A,0.082,8.2%
True,0.082,8.2%
(,0.072,7.2%
,0.072,7.2%


  -- True: stimulate vs baseline --


Token,Probability,Distribution
True,0.140,14.0%
,0.084,8.4%
(,0.058,5.8%
A,0.058,5.8%
,0.045,4.5%
Token,Probability,Distribution
True,0.226,22.6%
A,0.057,5.7%
,0.050,5.0%
The,0.044,4.4%


  -- False: ablate vs baseline --


...

  -- False: ablate vs baseline --


Token,Probability,Distribution
True,0.138,13.8%
,0.083,8.3%
A,0.057,5.7%
The,0.051,5.1%
(,0.051,5.1%
Token,Probability,Distribution
True,0.094,9.4%
,0.083,8.3%
(,0.064,6.4%
A,0.064,6.4%


  -- False: stimulate vs baseline --


Token,Probability,Distribution
True,0.138,13.8%
,0.083,8.3%
A,0.057,5.7%
The,0.051,5.1%
(,0.051,5.1%
Token,Probability,Distribution
True,0.428,42.8%
False,0.157,15.7%
,0.045,4.5%
,0.031,3.1%



CROSS-FEATURE SUMMARY (delta = gap change vs same-prompt baseline)
  Feature                T-abl    T-stim     F-abl    F-stim
------------------------------------------------------------
  (16,21,14143)        +0.1250   +0.6250   +0.2500   +0.7500
  (20,21,12997)        +0.1250   -0.2500   +0.2500   -0.1250
  (24,21,4640)         +0.0000   -0.7500   +0.0000   -1.1250
  (18,21,15770)        -0.2500   +1.8750   -0.2500   +2.0000
  (17,21,6448)         +0.0000   -0.3750   +0.1250   -0.2500


- All five interventions still sit at pos = 21, i.e. a single narrow slot aligned with response template / delimiter (Answer:-type) traffic. Shared geometry there makes correlated dynamics plausible even when directions differ row by row.
- Attributed rank still does not line up naively with ablation intuition: for ranks 1–2 ((16,21,14143), (20,21,12997)), T-abl is +0.125 — zeroing features the graph ranked at the top widens True − False on the true wording, opposite of “this unit is what pushes the affirmative verdict.”
- Effect size splits by probe: Ablations stay in a moderate band here (mostly −0.25 … +0.25 on this table). Stimulation (10 × baseline) is often much larger: e.g. (18,21,15770) shifts T-stim +1.875 and F-stim +2.000, while (24,21,4640) hits T-stim −0.750 / F-stim −1.125. So “small effects only” understates 10 × a; it pressures the gap more like a stress test than a gentle nudge.
- Treat ablate and stim as asymmetric probes. They need not behave like negatives of each other: (18,21,15770) pairs negative T-abl with strongly positive T-stim, and (20,21,12997) flips sign between T-abl and T-stim. That pattern fits nonlinear routing better than “one causal lever flipped two ways.” (24,21,4640) still looks mostly stim-driven on the gap (ablate flat on both prompts).

### 6b -- Sign-flip features from Section 5e: causal test

Section 5e found many features whose signed `True - False` influence *flipped* between the true and false claim. This cell uses the **top 5** by |Delta| (from `rows_5e` when available).

**Interventions** match **6a** and Anthropic `attribution_targets_demo.ipynb`: **ablate to 0**, **stimulate to `10 * a`** on the relevant prompt.

After the loop, a **cross-feature summary** aggregates `T-abl`, `T-stim`, `F-abl`, `F-stim`.

> If you re-run Section 5e with a new prompt, rely on `rows_5e[:5]` or refresh the fallback list in the code.



In [40]:
import torch
from functools import partial
from circuit_tracer.utils.demo_utils import display_topk_token_predictions

display_topk = partial(display_topk_token_predictions, tokenizer=tokenizer)

try:
    SIGNFLIP_FEATURES = [(L, P, F) for L, P, F, *_ in rows_5e[:5]]
except NameError:
    SIGNFLIP_FEATURES = [
        (6, 14, 4963),
        (1, 14, 12924),
        (10, 13, 3694),
        (0, 15, 8167),
        (1, 15, 12610),
    ]

STIM_MUL = 10.0

idx_true = tokenizer.encode(TOKEN_TRUE, add_special_tokens=False)[-1]
idx_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
key_tokens = [(TOKEN_TRUE, idx_true), (TOKEN_FALSE, idx_false)]


def _gap(logits):
    last = logits.squeeze(0)[-1].float()
    return (last[idx_true] - last[idx_false]).item()


with torch.inference_mode():
    _, act_prompt_T = model.get_activations(PROMPT, sparse=False)
    _, act_prompt_F = model.get_activations(PROMPT_FALSE_CLAIM, sparse=False)
    base_true_logits, _ = model.feature_intervention(PROMPT, [])
    base_false_logits, _ = model.feature_intervention(PROMPT_FALSE_CLAIM, [])

gap_base_true = _gap(base_true_logits)
gap_base_false = _gap(base_false_logits)

summary_rows = []

for feat_idx, (L, P, F) in enumerate(SIGNFLIP_FEATURES, start=1):
    label = f"({L},{P},{F})"
    a_T = float(act_prompt_T[L, P, F].item())
    a_F = float(act_prompt_F[L, P, F].item())
    stim_T = STIM_MUL * a_T
    stim_F = STIM_MUL * a_F

    print("\n" + "=" * 60)
    print(f"Feature {feat_idx}/5: {label}")
    print(f"  baseline activation  a_true={a_T:+.6f}  a_false={a_F:+.6f}  |  stim target ({STIM_MUL:g}x) -> {stim_T:+.6f}, {stim_F:+.6f}")
    print("=" * 60)

    with torch.inference_mode():
        abl_t_logits = model.feature_intervention(PROMPT, [(L, P, F, 0.0)])[0]
        stim_t_logits = model.feature_intervention(PROMPT, [(L, P, F, stim_T)])[0]
        abl_f_logits = model.feature_intervention(PROMPT_FALSE_CLAIM, [(L, P, F, 0.0)])[0]
        stim_f_logits = model.feature_intervention(PROMPT_FALSE_CLAIM, [(L, P, F, stim_F)])[0]

    gaps = {
        "base_T": gap_base_true,
        "abl_T": _gap(abl_t_logits),
        "stim_T": _gap(stim_t_logits),
        "base_F": gap_base_false,
        "abl_F": _gap(abl_f_logits),
        "stim_F": _gap(stim_f_logits),
    }

    for name, (lbl, base_key) in [
        ("base_T", ("True  baseline", None)),
        ("abl_T", ("True  ablate", "base_T")),
        ("stim_T", ("True  stimulate", "base_T")),
        ("base_F", ("False baseline", None)),
        ("abl_F", ("False ablate", "base_F")),
        ("stim_F", ("False stimulate", "base_F")),
    ]:
        d = f"{gaps[name] - gaps[base_key]:+.4f}" if base_key else "   --"
        print(f"  {lbl:<22}  gap={gaps[name]:>+8.4f}  delta={d}")

    summary_rows.append((label, gaps))

    print("\n  -- True: ablate vs baseline --")
    display_topk(PROMPT, base_true_logits, abl_t_logits, key_tokens=key_tokens)

    print("  -- True: stimulate vs baseline --")
    display_topk(PROMPT, base_true_logits, stim_t_logits, key_tokens=key_tokens)

    print("  -- False: ablate vs baseline --")
    display_topk(PROMPT_FALSE_CLAIM, base_false_logits, abl_f_logits, key_tokens=key_tokens)

    print("  -- False: stimulate vs baseline --")
    display_topk(PROMPT_FALSE_CLAIM, base_false_logits, stim_f_logits, key_tokens=key_tokens)

print("\n" + "=" * 78)
print("CROSS-FEATURE SUMMARY (delta = gap change vs same-prompt baseline)")
print("=" * 78)
print(f'  {"Feature":<18}  {"T-abl":>8}  {"T-stim":>8}  {"F-abl":>8}  {"F-stim":>8}')
print("-" * 60)
for lbl, g in summary_rows:
    print(
        f"  {lbl:<18}"
        f'  {g["abl_T"] - g["base_T"]:>+8.4f}'
        f'  {g["stim_T"] - g["base_T"]:>+8.4f}'
        f'  {g["abl_F"] - g["base_F"]:>+8.4f}'
        f'  {g["stim_F"] - g["base_F"]:>+8.4f}'
    )

cleanup_cuda()




Feature 1/5: (6,14,4963)
  baseline activation  a_true=+10.000000  a_false=+11.562500  |  stim target (10x) -> +100.000000, +115.625000
  True  baseline          gap= +1.5000  delta=   --
  True  ablate            gap= +1.5000  delta=+0.0000
  True  stimulate         gap= +1.5000  delta=+0.0000
  False baseline          gap= +1.2500  delta=   --
  False ablate            gap= +1.2500  delta=+0.0000
  False stimulate         gap= +1.3750  delta=+0.1250

  -- True: ablate vs baseline --


Token,Probability,Distribution
True,0.140,14.0%
,0.084,8.4%
(,0.058,5.8%
A,0.058,5.8%
,0.045,4.5%
Token,Probability,Distribution
True,0.141,14.1%
,0.085,8.5%
(,0.058,5.8%
A,0.058,5.8%


  -- True: stimulate vs baseline --


Token,Probability,Distribution
True,0.140,14.0%
,0.084,8.4%
(,0.058,5.8%
A,0.058,5.8%
,0.045,4.5%
Token,Probability,Distribution
True,0.115,11.5%
,0.089,8.9%
A,0.070,7.0%
(,0.062,6.2%


  -- False: ablate vs baseline --


...

  -- False: ablate vs baseline --


Token,Probability,Distribution
True,0.138,13.8%
,0.083,8.3%
A,0.057,5.7%
The,0.051,5.1%
(,0.051,5.1%
Token,Probability,Distribution
True,0.139,13.9%
,0.084,8.4%
The,0.051,5.1%
(,0.051,5.1%


  -- False: stimulate vs baseline --


Token,Probability,Distribution
True,0.138,13.8%
,0.083,8.3%
A,0.057,5.7%
The,0.051,5.1%
(,0.051,5.1%
Token,Probability,Distribution
True,0.139,13.9%
,0.084,8.4%
(,0.051,5.1%
A,0.051,5.1%



CROSS-FEATURE SUMMARY (delta = gap change vs same-prompt baseline)
  Feature                T-abl    T-stim     F-abl    F-stim
------------------------------------------------------------
  (6,14,4963)          +0.0000   +0.0000   +0.0000   +0.1250
  (1,14,12924)         +0.0000   +0.0000   +0.0000   +0.0000
  (10,13,3694)         +0.0000   +0.0000   +0.0000   +0.1250
  (0,15,8167)          +0.0000   -0.1250   +0.0000   +0.0000
  (1,15,12610)         +0.0000   +0.0000   +0.0000   +0.0000


- Location matters: Unlike §6a, these hooks monitor positions 13–15 (on predicate/wording tokens) instead of the Answer: slot (around pos 21) that dominated attribution rank. This geographic difference helps explain why effects on the True − False gap are weaker at the last position when attention is frozen.
- Mostly zeros: For four of the five features, most cells in the table show exactly ±0. At this probe’s sensitivity, the sign-flippers appear largely causally idle; even sizeable stimulation (10 × a, where a ≫ 0) has little to no impact on the main gap unless it directly influences the contrast logits.
- Movement is subtle: When there is an effect, it’s small (~0.125). The pairs (6,14,4963) and (10,13,3694) only show a +0.125 increase for F-stim (false prompt, stimulated), while (0,15,8167) is the lone mover for the true prompt, nudging T-stim to −0.125 (a slight gap shrink). These are hints, not decisive circuit evidence.
- Ablations change little: T-abl and F-abl register exactly 0 everywhere. Removing these units leaves the True − False gap unaffected, even in cases where §5e found them important—highlighting that a strong contrast in attribution doesn’t guarantee a gap shift under knockout.
- Compared to §6a: The verdict-slot features in §6a exerted stronger effects. Here, the §5e sign-flippers are weak causal levers for the same readout, even under 10× stimulation. The pattern is dominated by inactivity, punctuated by a few small positive shifts on the false prompt and a slight shrink on true.

### 6c -- Geometry gallery features from Section 5f: causal test

Section 5f examined the five hand-labelled gallery features that appeared in both attribution graphs:

| Nickname | (L, P, F) | Neuronpedia theme |
|---|---|---|
| Sides and vertices barista | (8, 12, 7823)  | geometry / equations |
| Faithful triangle word     | (5,  5, 15376) | triangle, codon |
| Three-ish Trojan horse     | (7,  5, 11752) | counting / football |
| Trig cheat-sheet           | (7,  7, 12658) | symbols / geometry |
| Shape cafeteria            | (7,  5,  9008) | polygons / shapes |

Same **ablate / 10x-stim** recipe as **6a** and **6b** (`get_activations` baselines per prompt). Compare whether geometry-tagged hooks move the `True - False` gap more than the delimiter-heavy top-five in **6a** or the sign-flip set in **6b**.



In [41]:
import torch
from functools import partial
from circuit_tracer.utils.demo_utils import display_topk_token_predictions

display_topk = partial(display_topk_token_predictions, tokenizer=tokenizer)

try:
    GALLERY_FEATURES = [(L, P, F, name) for name, L, P, F, *_ in rows_5f]
except NameError:
    GALLERY_FEATURES = [
        (8, 12, 7823, "Sides & vertices barista"),
        (5, 5, 15376, "Faithful triangle word"),
        (7, 5, 11752, "Three-ish Trojan horse"),
        (7, 7, 12658, "Trig cheat-sheet"),
        (7, 5, 9008, "Shape cafeteria"),
    ]

STIM_MUL = 10.0

idx_true = tokenizer.encode(TOKEN_TRUE, add_special_tokens=False)[-1]
idx_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
key_tokens = [(TOKEN_TRUE, idx_true), (TOKEN_FALSE, idx_false)]


def _gap(logits):
    last = logits.squeeze(0)[-1].float()
    return (last[idx_true] - last[idx_false]).item()


with torch.inference_mode():
    _, act_prompt_T = model.get_activations(PROMPT, sparse=False)
    _, act_prompt_F = model.get_activations(PROMPT_FALSE_CLAIM, sparse=False)
    base_true_logits, _ = model.feature_intervention(PROMPT, [])
    base_false_logits, _ = model.feature_intervention(PROMPT_FALSE_CLAIM, [])

gap_base_true = _gap(base_true_logits)
gap_base_false = _gap(base_false_logits)

summary_rows = []

for L, P, F, nickname in GALLERY_FEATURES:
    feat_label = f"{nickname} ({L},{P},{F})"
    a_T = float(act_prompt_T[L, P, F].item())
    a_F = float(act_prompt_F[L, P, F].item())
    stim_T = STIM_MUL * a_T
    stim_F = STIM_MUL * a_F

    print("\n" + "=" * 66)
    print(feat_label)
    print(f"  baseline activation  a_true={a_T:+.6f}  a_false={a_F:+.6f}  |  stim target ({STIM_MUL:g}x) -> {stim_T:+.6f}, {stim_F:+.6f}")
    print("=" * 66)

    with torch.inference_mode():
        abl_t_logits = model.feature_intervention(PROMPT, [(L, P, F, 0.0)])[0]
        stim_t_logits = model.feature_intervention(PROMPT, [(L, P, F, stim_T)])[0]
        abl_f_logits = model.feature_intervention(PROMPT_FALSE_CLAIM, [(L, P, F, 0.0)])[0]
        stim_f_logits = model.feature_intervention(PROMPT_FALSE_CLAIM, [(L, P, F, stim_F)])[0]

    gaps = {
        "base_T": gap_base_true,
        "abl_T": _gap(abl_t_logits),
        "stim_T": _gap(stim_t_logits),
        "base_F": gap_base_false,
        "abl_F": _gap(abl_f_logits),
        "stim_F": _gap(stim_f_logits),
    }

    for name, (lbl, base_key) in [
        ("base_T", ("True  baseline", None)),
        ("abl_T", ("True  ablate", "base_T")),
        ("stim_T", ("True  stimulate", "base_T")),
        ("base_F", ("False baseline", None)),
        ("abl_F", ("False ablate", "base_F")),
        ("stim_F", ("False stimulate", "base_F")),
    ]:
        d = f"{gaps[name] - gaps[base_key]:+.4f}" if base_key else "   --"
        print(f"  {lbl:<22}  gap={gaps[name]:>+8.4f}  delta={d}")

    summary_rows.append((nickname, gaps))

    print("\n  -- True: ablate vs baseline --")
    display_topk(PROMPT, base_true_logits, abl_t_logits, key_tokens=key_tokens)

    print("  -- True: stimulate vs baseline --")
    display_topk(PROMPT, base_true_logits, stim_t_logits, key_tokens=key_tokens)

    print("  -- False: ablate vs baseline --")
    display_topk(PROMPT_FALSE_CLAIM, base_false_logits, abl_f_logits, key_tokens=key_tokens)

    print("  -- False: stimulate vs baseline --")
    display_topk(PROMPT_FALSE_CLAIM, base_false_logits, stim_f_logits, key_tokens=key_tokens)

print("\n" + "=" * 78)
print("CROSS-FEATURE SUMMARY (delta = gap change vs same-prompt baseline)")
print("=" * 78)
print(f'  {"Feature":<32}  {"T-abl":>8}  {"T-stim":>8}  {"F-abl":>8}  {"F-stim":>8}')
print("-" * 74)
for nick, g in summary_rows:
    print(
        f"  {nick:<32}"
        f'  {g["abl_T"] - g["base_T"]:>+8.4f}'
        f'  {g["stim_T"] - g["base_T"]:>+8.4f}'
        f'  {g["abl_F"] - g["base_F"]:>+8.4f}'
        f'  {g["stim_F"] - g["base_F"]:>+8.4f}'
    )

cleanup_cuda()




Sides & vertices barista (8,12,7823)
  baseline activation  a_true=+21.125000  a_false=+21.125000  |  stim target (10x) -> +211.250000, +211.250000
  True  baseline          gap= +1.5000  delta=   --
  True  ablate            gap= +1.5000  delta=+0.0000
  True  stimulate         gap= +1.3750  delta=-0.1250
  False baseline          gap= +1.2500  delta=   --
  False ablate            gap= +1.2500  delta=+0.0000
  False stimulate         gap= +1.2500  delta=+0.0000

  -- True: ablate vs baseline --


Token,Probability,Distribution
True,0.140,14.0%
,0.084,8.4%
(,0.058,5.8%
A,0.058,5.8%
,0.045,4.5%
Token,Probability,Distribution
True,0.139,13.9%
,0.084,8.4%
(,0.058,5.8%
A,0.058,5.8%


  -- True: stimulate vs baseline --


Token,Probability,Distribution
True,0.140,14.0%
,0.084,8.4%
(,0.058,5.8%
A,0.058,5.8%
,0.045,4.5%
Token,Probability,Distribution
True,0.140,14.0%
A,0.075,7.5%
,0.075,7.5%
(,0.066,6.6%


  -- False: ablate vs baseline --


...

  -- False: ablate vs baseline --


Token,Probability,Distribution
True,0.138,13.8%
,0.083,8.3%
A,0.057,5.7%
The,0.051,5.1%
(,0.051,5.1%
Token,Probability,Distribution
True,0.137,13.7%
,0.083,8.3%
A,0.057,5.7%
The,0.050,5.0%


  -- False: stimulate vs baseline --


Token,Probability,Distribution
True,0.138,13.8%
,0.083,8.3%
A,0.057,5.7%
The,0.051,5.1%
(,0.051,5.1%
Token,Probability,Distribution
True,0.139,13.9%
,0.084,8.4%
A,0.058,5.8%
The,0.051,5.1%



CROSS-FEATURE SUMMARY (delta = gap change vs same-prompt baseline)
  Feature                              T-abl    T-stim     F-abl    F-stim
--------------------------------------------------------------------------
  Sides & vertices barista           +0.0000   -0.1250   +0.0000   +0.0000
  Faithful △ word                    +0.0000   +0.0000   +0.0000   +0.1250
  Three-ish Trojan horse             +0.0000   +0.0000   +0.0000   +0.1250
  Trig cheat-sheet                   +0.0000   +0.0000   +0.0000   +0.0000
  Shape cafeteria                    +0.0000   +0.1250   +0.0000   +0.2500


- Gallery hooks primarily target content tokens (positions 5–12), rather than the "Answer:" slot (around position 21). This is consistent with §5f, which covers geometry-anchored directions—like triangle, sides, and trigonometry features. Note that changes in the logit gap at the "Answer:" position may still be controlled upstream by template heads, so small or inconsistent effects here are not surprising from a geometric perspective.
- With zero ablation, every T-abl and F-abl value remains exactly zero—disabling each labeled feature has no effect on logit(True) − logit(False). This suggests that these directions act as non-pivoting, single-channel knockout levers for this readout, even when their Neuronpedia tags appear thematically relevant.
- When features are stimulated to 10× their baseline value, we begin to see small effects: the logit gap can shift by Δ ≈ −0.125 to +0.25, indicating subtle but measurable changes. However, these shifts are exploratory in nature and do not provide strong evidence of direct causality.
- The effects are sparse and specific: For example, ‘Sides & vertices barista’ is the only feature to move the true prompt, reducing the gap (T-stim −0.125); ‘Faithful △’ and ‘Three-ish’ each only widen the gap for the false prompt (F-stim +0.125); ‘Trig cheat-sheet’ remains inactive throughout; and ‘Shape cafeteria’ is the most responsive with small increases (T-stim +0.125, F-stim +0.250), though the effects are still minor.
- In comparison to §6a, these five geometry-labeled features act as much weaker levers on the True−False gap than the most strongly attributed features at position 21, even with the same 10× stimulation. This supports a nuanced conclusion: “geometry-flavored” dashboard features rarely have dominant local causal effects on the verdict gap, even when they're present and tagged as relevant.

## Section 7 -- Conclusions

### 7.1 Behavior: first-token distribution and multi-step rollout

**Softmax at the delimiter** (Section 2) shows whether True and False are competitive next tokens, not merely present in the tail of the vocabulary. That matters because later methods attribute this same contrast target , so attribution is tied to whichever distribution the model exposes at that position.

**Greedy rollout** (Section 3) shows that decoding is **path-dependent**. Even when an early greedy step looks verdict-like (True-ish), subsequent greedy steps often continue high-probability instructional patterns. Further, fro prompt-2 (other notebok) the most-likely token continuation did not result in meaningful sequence.

---

### 7.2 Graph analysis: where influence concentrates, and what flips under negation

**Global influence ranking** (Section 5) compresses a large attributed subgraph into a short list of **(layer, pos, feat)** hooks ranked by aggregated influence on logits (including indirect paths weighted by logits). Practically, the reader-facing story is dominated by pos-21-ish features anchored near the **Answer:** delimiter region: the model allocates much of this contrast's attribution budget to **format / verdict-slot** machinery rather than to an obviously geometric token slice.

**Contrastive attribution** (Section 5c–5e) asks a sharper question than the single-graph table: **which features systematically reverse** their attributed push between a mathematically true statement and a negated statement (same scaffolding, flipped predicate semantics). §5e shows that many intersection features flip sign inside the attribution construction, with interpretable-but-noisy correlates under Neuronpedia-style labels.

**Gallery alignment** (Section 5f) grounds the abstract graph in hand-labelled geometry directions wherever they appear inside both subgraphs—without insisting on flip sign. Several rows behave like plausible triangle / notation / polygons scouts on content tokens (pos ∈ [5, 18]), complementing—but not replacing—the dominant delimiter-focused picture.

**Highlight:** *the attribution graph yields a nuanced map: abundant **template-aligned** influence alongside **semantically keyed** contrasts that reorganize across negation.*

---

### 7.3 Interventions: correlation vs causation under a fixed probe



**§6a (Top Five Attributed Features):** All five top features are concentrated at **position 21**—the verdict or template slot. Ablating these features leads to only modest shifts in the True−False gap, while stimulation shows much larger and often non-opposite effects, reflecting the nonlinear nature of the probes. Rather than acting as simple “on-off switches” for the verdict, these features seem mechanically coupled to this token band: their influence is real, but not neatly directionally causal according to rank alone.


**§6b (Sign-Flip Features from §5e):** Here, most features have little effect on the metric: ablations do not produce measurable changes, and stimulation yields only subtle nudges—such as +0.125 on **F-stim** in two cases and −0.125 on **T-stim** in another. These are very minor (∼0.125 logit) and do not represent strong levers. That these features appear at positions 13–15, differing from §6a's focus, suggests a weaker link to the logits at the **Answer:** slot.
 
 
**§6c (Geometry Gallery from §5f):** For geometry-labeled features, ablations leave the gap unchanged, but stimulating at **10× baseline** produces a few small, scattered effects: for instance, one row with **T-stim −0.125**, two rows with **F-stim** +0.125, and **Shape cafeteria** with +0.125 (true) and +0.25 (false) stimulation. The **Trig cheat-sheet** remains inactive. In summary, Neuronpedia-style "geometry" hooks provide only weak, single-feature control over the True−False contrast here—much less than the delimiter features in §6a—though with some signs of impact when stimulation is increased.


---

### 7.4 Key takaways

1. **Model verdicts involve multiple mechanisms:** The model’s decision between “True” and “False” is the result of several distinct processes. Looking at the probability assigned to each token at the answer slot (via the softmax), running various sampling strategies, and following the most likely next-step predictions (“greedy continuation”) can each produce their own distinct outcomes. These methods reveal that “the model’s answer” is not always a single, clear thing, but depends on how you measure or probe it.
2. **Attribution for True − False reflects credit allocation, but not full causality:** When we analyze which internal features contribute most to increasing the gap between “True” and “False” answers, attribution methods highlight **where** in the network this distinction is sharpened. Usually, this concentration of influence happens around areas managing answer formatting, not just on features directly encoding geometric or mathematical content. Importantly, attribution shows correlation (where the model “credits” an effect), not guaranteed causality.
3. **Negating statements exposes model organization, not simple control switches:** Some internal features sharply reverse their attributed influence when the prompt is negated (e.g., swapping “is” for “is not”). These sign-flipping features are interesting hypotheses about what the model may use to distinguish language contrasts, but simply finding a sign flip in attribution does not mean toggling that feature will reliably flip the model’s output in practice.
4. **Geometry-labeled features are interpretable but rarely strong levers:** Even when features are clearly tagged as related to “geometry” (see Section 5f) and present in the model’s computations, direct intervention experiments show that turning them off (“ablations”) often has no effect on the True-False gap, while stimulating them far above their natural activity (“10× stimulation”) only causes small, scattered shifts (typically ±0.125 logits, reaching up to ~+0.25 in rare cases). This teaches us that it’s not enough for a feature to be labeled as relevant: labels suggest possibility, but don’t guarantee that feature alone can control the model’s answer.

